# 《Re-Imag(in)ing Price Trends》复现 Notebook（含章节说明）

按 **数据 → 环境 → 清洗 → 模型/数据集 → 训练 → 外样本预测与回测 → 拓展分析 → 论文图表** 组织。**请从上到下依次运行**；跳过前置单元会导致后续缺少变量报错。

## 流程概览

| 章节 | 内容 |
|------|------|
| 零 | 原始 CSV 速览（可选） |
| 一 | 运行环境（OpenMP / PyTorch） |
| 二 | 数据清洗、周期持有期收益、`EWMA_vol`、缓存 |
| 三 | CNN、`StockImageDataset`、周期采样与回测辅助函数 |
| 四 | 训练/验证划分与多 horizon 训练 |
| 五 | 测试集集成预测与多空回测 |
| 六 | 导出 PPT 素材（可选） |
| 七 | Fama–French 因子回归（需联网） |
| 八 | Saliency 可解释性 |
| 九 | Figure 6/7/8、Table I |
| 十 | 分类指标与夏普排行 |

---


## 零、原始数据速览（可选）

读取 `DATA_PATH`（默认指向项目 raw/OHLC_92_24.csv），查看前几行，确认路径与列名正确。

In [ ]:
import pandas as pd
import numpy as np

data_path = globals().get('DATA_PATH', '../raw/OHLC_92_24.csv')
df = pd.read_csv(data_path)
df.head()

### 本单元做什么

打印 `DataFrame` 的结构、`dtype` 与行数（内存占用），用于快速体检。

In [ ]:
df.info()

### 本单元做什么

数值列描述性统计（`describe`），检查收益率与价格量级是否异常。

In [ ]:
df.describe()

## 一、运行环境与深度学习后端

设置 OpenMP、导入 **PyTorch**、定义 `device`（优先 MPS）、创建 `outputs/` 下缓存与模型目录。

### 冒烟测试 `SMOKE_TEST`

下一单元中 **`SMOKE_TEST = True`** 时为**冒烟子样本**模式（仍为 `read_csv` 截断 + 日期窗 + permno 子集；默认参数已加大以利于测试窗、回测与 FF 回归有足够观测）；默认只跑 **I5**。权重与清洗缓存写入 **`outputs/models_smoke`**、**`outputs/cache_smoke`**，**不会覆盖**全样本产物。若要完整复现前请将 **`SMOKE_TEST` 改为 `False`**。

若报错里仍出现 **`rename_dict` / `factors_clean`**，说明 Kernel 里是旧版 `run_factor_regression`，请 **Restart Kernel** 后从环境与数据单元依次重跑。**


In [ ]:
import os
import platform
from pathlib import Path

# macOS/conda 下偶发 OpenMP 冲突；Linux 云服务器默认不启用该 workaround。
if platform.system() == "Darwin":
    os.environ.setdefault("KMP_DUPLICATE_LIB_OK", "TRUE")

import gc
import pickle
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from tqdm.auto import tqdm

# 路径配置：可在云服务器用环境变量覆盖，例如 DATA_PATH=/mnt/data/OHLC_92_24.csv OUTPUT_DIR=/mnt/output
_CWD = Path.cwd().resolve()
if (_CWD / "raw").exists() and (_CWD / "scripts").exists():
    PROJECT_ROOT = _CWD
elif _CWD.name == "scripts" and _CWD.parent.name == "Final":
    PROJECT_ROOT = _CWD.parent
elif (_CWD / "Final" / "raw").exists():
    PROJECT_ROOT = _CWD / "Final"
else:
    PROJECT_ROOT = _CWD.parent if _CWD.name == "scripts" else _CWD

DATA_PATH = os.environ.get("DATA_PATH", str(PROJECT_ROOT / "raw" / "OHLC_92_24.csv"))
OUTPUT_DIR = os.environ.get("OUTPUT_DIR", str(PROJECT_ROOT / "outputs"))
os.makedirs(OUTPUT_DIR, exist_ok=True)

# 设备选择：云服务器 CUDA 优先；本地 Mac 可用 MPS；否则 CPU。
def select_device():
    if torch.cuda.is_available():
        return torch.device("cuda")
    if hasattr(torch.backends, "mps") and torch.backends.mps.is_available():
        return torch.device("mps")
    return torch.device("cpu")

device = select_device()
PIN_MEMORY = device.type == "cuda"
# 多核配置：默认开小步并行（2核），可通过环境变量覆盖
_default_workers = 2
DATALOADER_NUM_WORKERS = int(os.environ.get("DATALOADER_NUM_WORKERS", _default_workers))
_DEFAULT_THREADS = int(os.environ.get("CPU_THREADS", "2"))
os.environ["OMP_NUM_THREADS"] = str(_DEFAULT_THREADS)
os.environ.setdefault("MKL_NUM_THREADS", str(_DEFAULT_THREADS))
os.environ.setdefault("OPENBLAS_NUM_THREADS", str(_DEFAULT_THREADS))
os.environ.setdefault("NUMEXPR_NUM_THREADS", str(_DEFAULT_THREADS))
torch.set_num_threads(_DEFAULT_THREADS)
if hasattr(torch, "set_num_interop_threads"):
    torch.set_num_interop_threads(_DEFAULT_THREADS)

# 稳定性优先：默认关闭 torch.compile，需手动设置 TORCH_COMPILE=1 才启用
TORCH_COMPILE = os.environ.get("TORCH_COMPILE", "0").strip().lower() in {"1", "true", "yes", "on"}
PRED_BATCH_SIZE = int(os.environ.get("PRED_BATCH_SIZE", "1024"))
USE_FP16_INFER = os.environ.get("USE_FP16_INFER", "1").strip().lower() in {"1", "true", "yes", "on"}
print(
    f"Using device: {device} | DataLoader workers={DATALOADER_NUM_WORKERS} | pin_memory={PIN_MEMORY} | "
    f"threads={_DEFAULT_THREADS} | torch.compile={TORCH_COMPILE} | pred_batch={PRED_BATCH_SIZE} | fp16={USE_FP16_INFER}"
)


def maybe_compile(model: nn.Module) -> nn.Module:
    if not TORCH_COMPILE:
        return model
    compile_mode = os.environ.get("TORCH_COMPILE_MODE", "default")
    print(f"[INFO] torch.compile enabled (mode={compile_mode})")
    return torch.compile(model, mode=compile_mode)


def make_dataloader(dataset, batch_size, shuffle=False, **kwargs):
    kwargs.setdefault("num_workers", DATALOADER_NUM_WORKERS)
    kwargs.setdefault("pin_memory", PIN_MEMORY)
    if kwargs["num_workers"] > 0:
        kwargs.setdefault("persistent_workers", True)
        kwargs.setdefault("prefetch_factor", 2)
    return DataLoader(dataset, batch_size=batch_size, shuffle=shuffle, **kwargs)

# =============================================================================
# SMOKE_TEST：极速冒烟 —— 只验证管线能否跑通（无数值/论文意义）
# 正式复现前请改为 False；为 True 时使用独立 cache/models 子目录，避免覆盖全样本缓存
# =============================================================================
SMOKE_TEST = False

if SMOKE_TEST:
    # 冒烟仍快于全量，但显著提高子样本规模，避免测试窗/因子回归因观测过少报错
    SMOKE_READ_CSV_NROWS = 6_000_000
    SMOKE_DATE_START = "2008-01-01"
    SMOKE_DATE_END = "2019-12-31"
    SMOKE_MAX_PERMNOS = 350
    # 只跑 (L, F) 任务列表；全量请 [(5, 5), (20, 20), (60, 60)]
    SMOKE_TASKS = [(5, 5)]
    SMOKE_MAX_EPOCHS = 18
    SMOKE_SEEDS = [42]
    SMOKE_BATCH_SIZE = 128
    SMOKE_NORM_SAMPLES = 10_240
    SMOKE_LIQUIDITY_TOP_N = 700
    SMOKE_FORCE_REBUILD = True
    MODEL_DIR = os.path.join(OUTPUT_DIR, 'models_smoke')
    CACHE_DIR = os.path.join(OUTPUT_DIR, 'cache_smoke')
else:
    SMOKE_READ_CSV_NROWS = None
    SMOKE_DATE_START = None
    SMOKE_DATE_END = None
    SMOKE_MAX_PERMNOS = None
    SMOKE_TASKS = [(5, 5), (20, 20), (60, 60)]
    SMOKE_MAX_EPOCHS = 50
    SMOKE_SEEDS = [42, 123, 456, 789, 101112]
    SMOKE_BATCH_SIZE = 128
    SMOKE_NORM_SAMPLES = 50_000
    SMOKE_LIQUIDITY_TOP_N = None
    SMOKE_FORCE_REBUILD = False
    MODEL_DIR = os.path.join(OUTPUT_DIR, 'models')
    CACHE_DIR = os.path.join(OUTPUT_DIR, 'cache')

os.makedirs(MODEL_DIR, exist_ok=True)
os.makedirs(CACHE_DIR, exist_ok=True)

if SMOKE_TEST:
    print("[SMOKE_TEST] ON — 使用", CACHE_DIR, "|", MODEL_DIR)
else:
    print("[SMOKE_TEST] OFF — 全样本复现模式")


## 二、数据清洗与标签构造

定义并运行 `load_and_preprocess_data`：统一列名、排序、缺失处理、复权价、**周期末向前持有期收益** `R_fut_*`、`EWMA_vol`、标签与可选缓存 `cleaned_data_*.pkl`。

**依赖**：原始 CSV；本节结束后应得到可在后续单元使用的 `df_clean`（以及缓存文件）。

In [ ]:
import ast
from datetime import datetime
import os
import gc
import matplotlib.pyplot as plt

# ==========================================
# 第一部分：数据预处理与特征构建 (Step 1 & 2)
# ==========================================
def load_and_preprocess_data(
    data_path,
    cache_dir=None,
    force_rebuild=False,
    liquidity_top_n=None,
    nrows=None,
    cache_extra_tag='',
):
    """读取、清洗数据，并计算复权价、因变量标签。

    liquidity_top_n:
        None — 不做月度市值筛选（全样本，与作者默认 stocks_for_train=\"all\" 一致）
        正整数 — 每月按 dlycap 保留市值前 N 的股票（历史 notebook 的 top1000 行为）
    nrows:
        传给 pd.read_csv 的最大行数；冒烟测试用大幅缩短读盘时间
    cache_extra_tag:
        附加在缓存文件名上，避免与全样本 pkl 冲突（如 '_smoke'）
    """
    liq_core = 'all' if liquidity_top_n is None else f'top{int(liquidity_top_n)}'
    liq_tag = liq_core + str(cache_extra_tag)
    cache_file = os.path.join(cache_dir, f'cleaned_data_{liq_tag}.pkl')
    if os.path.exists(cache_file) and not force_rebuild:
        print("====== 读取已缓存的清洗后数据 ======")
        return pd.read_pickle(cache_file)

    print("====== 开始处理原始数据 ======")

    # 1. 加载数据，并将列名统一转为小写
    if nrows is not None:
        print(f"pd.read_csv nrows={nrows} （冒烟模式截断）")
        df = pd.read_csv(data_path, nrows=int(nrows))
    else:
        df = pd.read_csv(data_path)
    df.columns = [col.lower() for col in df.columns]

    df['dlycaldt'] = pd.to_datetime(df['dlycaldt'])
    df = df.sort_values(by=['permno', 'dlycaldt']).reset_index(drop=True)

    # 剔除存在缺失值的行
    essential_cols = ['dlyopen', 'dlyhigh', 'dlylow', 'dlyclose', 'dlyvol', 'dlyret']
    df = df.dropna(subset=essential_cols)

    # 2. 基础过滤规则
    df = df[(df['dlyclose'] >= 1.0) & (df['dlyvol'] > 0)].copy()

    # ==========================
    # 性能优化：全面向量化计算，防止 OOM 内存爆炸与 Kernel 崩溃
    # ==========================
    print("计算复权价格 (向量化，对数收益累积)...")
    # 3. 对数收益累积与复权因子（与作者 equity_data.py L109-110 一致）
    df['log_ret'] = np.log(1 + df['dlyret'])
    df['cum_log_ret'] = df.groupby('permno')['log_ret'].cumsum(skipna=True)
    # AF[t] = exp(cum_log_ret[t] - cum_log_ret[首日])，保证首日 AF=1
    df['AF'] = np.exp(df['cum_log_ret'] - df.groupby('permno')['cum_log_ret'].transform('first'))
    df.drop(columns=['log_ret'], inplace=True)

    # 计算复权价格
    # 取首日真实价格做基准
    df['base_price'] = df.groupby('permno')['dlyclose'].transform('first')
    df['dlyclose_adj'] = df['base_price'] * df['AF']
    # 利用当日内的开、高、低相对收盘的比例，推算出复权的开、高、低
    df['dlyopen_adj'] = df['dlyclose_adj'] * (df['dlyopen'] / df['dlyclose'])
    df['dlyhigh_adj'] = df['dlyclose_adj'] * (df['dlyhigh'] / df['dlyclose'])
    df['dlylow_adj'] = df['dlyclose_adj'] * (df['dlylow'] / df['dlyclose'])
    df.drop(columns=['base_price'], inplace=True)

    print("计算未来累计收益和标签 (向量化)...")
    # 4. 用对数收益累积计算未来收益（与作者 equity_data.py L117-118 一致）
    # R_fut_F = exp(cum_log_ret[t+F] - cum_log_ret[t]) - 1
    for F in [5, 20, 60]:
        df[f'cum_log_ret_shift_{F}'] = df.groupby('permno')['cum_log_ret'].shift(-F)
        df[f'R_fut_{F}'] = np.exp(df[f'cum_log_ret_shift_{F}'] - df['cum_log_ret']) - 1
        df[f'Label_{F}'] = (df[f'R_fut_{F}'] > 0).astype(int)
        df.drop(columns=[f'cum_log_ret_shift_{F}'], inplace=True)
    df.drop(columns=['cum_log_ret'], inplace=True)

    if liquidity_top_n is not None:
        print(f"应用流动性筛选：每月保留市值前 {int(liquidity_top_n)} ...")
        df['year_month'] = df['dlycaldt'].dt.to_period('M')
        last_cap = df.groupby(['year_month', 'permno'])['dlycap'].last().reset_index()
        top_n = last_cap.sort_values(by=['year_month', 'dlycap'], ascending=[True, False]).groupby('year_month').head(int(liquidity_top_n))
        top_n['is_liquid'] = True
        df = df.merge(top_n[['year_month', 'permno', 'is_liquid']], on=['year_month', 'permno'], how='inner')
        df.drop(columns=['year_month', 'is_liquid'], inplace=True)
    else:
        print("流动性筛选：关闭（全样本）")

    # 释放无用内存
    gc.collect()

    df.to_pickle(cache_file)
    print("====== 数据清洗完成并缓存 ======")
    return df


def subsample_df_for_smoke(df):
    """在清洗结果上再裁日历 + 随机股票子集（仅 SMOKE_TEST）。"""
    if not globals().get("SMOKE_TEST", False):
        return df
    d0 = pd.Timestamp(SMOKE_DATE_START)
    d1 = pd.Timestamp(SMOKE_DATE_END)
    d = df[(df["dlycaldt"] >= d0) & (df["dlycaldt"] <= d1)].copy()
    perm = d["permno"].unique()
    rng = np.random.default_rng(42)
    cap = int(SMOKE_MAX_PERMNOS)
    if len(perm) > cap:
        keep = rng.choice(perm, size=cap, replace=False)
        d = d[d["permno"].isin(keep)].copy()
    print(
        f"[SMOKE] 子样本 shape={d.shape} permnos={d['permno'].nunique()} "
        f"dates={d['dlycaldt'].min()} .. {d['dlycaldt'].max()}"
    )
    return d


# 样本池：正式跑时 None=全样本；冒烟跑使用上面的 SMOKE_LIQUIDITY_TOP_N
LIQUIDITY_TOP_N = None  # 例如 1000 表示月度 top1000（仅 SMOKE_TEST=False 时生效）

_df_kwargs = dict(
    data_path=DATA_PATH,
    cache_dir=CACHE_DIR,
    liquidity_top_n=(SMOKE_LIQUIDITY_TOP_N if SMOKE_TEST else LIQUIDITY_TOP_N),
    nrows=SMOKE_READ_CSV_NROWS,
    cache_extra_tag="_smoke" if SMOKE_TEST else "",
    force_rebuild=(SMOKE_FORCE_REBUILD if SMOKE_TEST else False),
)

df_clean = load_and_preprocess_data(**_df_kwargs)
df_clean = subsample_df_for_smoke(df_clean)


### K 线图为何不单独占一个「预生成大数据集」单元？

K 线张量在 `StockImageDataset.__getitem__` 中按 **(permno, 周期末日)** 即时渲染；预先展开全样本会极端占内存。模型训练单元会按需读取清洗后的面板数据。

## 三、CNN 模型、`StockImageDataset` 与回测辅助函数

包含：`MODEL_CFG`、`PREDICT_HORIZON_TO_TRADE_FREQ`、`PERIODS_PER_YEAR`、`StockImageDataset`（过滤有效 `R_fut` / `EWMA_vol`）、CNN 及 **I5/I20/I60** 构建函数、`enrich_rebalance_panel_with_daily_baselines`、`evaluate_strategy` 等。

**依赖**：第二节的 `df_clean`；须先运行第一节以加载 `torch`。

In [ ]:
# ==========================================
# 第二部分：K线图像生成与张量提取 (Step 3)
# ==========================================

import random
import torch.nn.functional as F
from torch.utils.data import Dataset
from PIL import Image, ImageDraw

# 配置三大模型的图像尺寸与架构
MODEL_CFG = {
    5:  {'W': 15,  'H': 32, 'H_ohlc': 25, 'H_vol': 6},
    20: {'W': 60,  'H': 64, 'H_ohlc': 51, 'H_vol': 12},
    60: {'W': 180, 'H': 96, 'H_ohlc': 76, 'H_vol': 19},
}

# 与作者 dgp_config.FREQ_DICT 一致：预测 horizon -> 训练/取样频率
PREDICT_HORIZON_TO_TRADE_FREQ = {5: 'week', 20: 'month', 60: 'quarter'}

# 与作者 Portfolio.portfolio_res_summary 一致：每期持有收益年化用的每年期数
PERIODS_PER_YEAR = {5: 52, 20: 12, 60: 4}


def enrich_rebalance_panel_with_daily_baselines(df_sparse, df_daily, F_horizon):
    """在日频面板上计算动量/均线/短期反转，再 merge 回周期末稀疏面板（与训练取样一致）。"""
    d = df_daily.sort_values(['permno', 'dlycaldt']).copy()
    d['mom_score'] = d.groupby('permno')['dlyclose_adj'].pct_change(F_horizon)
    ma_fast = d.groupby('permno')['dlyclose_adj'].transform(
        lambda x: x.rolling(max(2, F_horizon // 2), min_periods=1).mean()
    )
    ma_slow = d.groupby('permno')['dlyclose_adj'].transform(
        lambda x: x.rolling(max(20, F_horizon), min_periods=1).mean()
    )
    d['ma_score'] = (ma_fast / ma_slow - 1).fillna(0)
    d['reversal_score'] = -d.groupby('permno')['dlyclose_adj'].pct_change(5)
    sub = d[['permno', 'dlycaldt', 'mom_score', 'ma_score', 'reversal_score']]
    out = df_sparse.merge(sub, on=['permno', 'dlycaldt'], how='left')
    out['mom_score'] = out['mom_score'].fillna(0)
    out['ma_score'] = out['ma_score'].fillna(0)
    out['reversal_score'] = out['reversal_score'].fillna(0)
    return out


def merge_linear_features_from_daily(df_sparse, df_daily):
    """Figure7 Linear 所需特征：在日频上 rolling/shift，再在调仓日 merge。"""
    d = df_daily.sort_values(['permno', 'dlycaldt']).copy()
    d['ret_past_1'] = d.groupby('permno')['dlyret'].shift(1)
    d['ret_past_5'] = d.groupby('permno')['dlyret'].transform(
        lambda x: x.shift(1).rolling(5, min_periods=1).mean()
    )
    d['ret_past_20'] = d.groupby('permno')['dlyret'].transform(
        lambda x: x.shift(1).rolling(20, min_periods=1).mean()
    )
    d['vol_past_5'] = d.groupby('permno')['dlyvol'].transform(
        lambda x: x.shift(1).rolling(5, min_periods=1).mean()
    )
    d['vol_past_20'] = d.groupby('permno')['dlyvol'].transform(
        lambda x: x.shift(1).rolling(20, min_periods=1).mean()
    )
    d['high_low_ratio'] = d['dlyhigh'] / (d['dlylow'] + 1e-12)
    d['close_open_ratio'] = d['dlyclose'] / (d['dlyopen'] + 1e-12)
    cols = [
        'permno',
        'dlycaldt',
        'ret_past_1',
        'ret_past_5',
        'ret_past_20',
        'vol_past_5',
        'vol_past_20',
        'high_low_ratio',
        'close_open_ratio',
    ]
    return df_sparse.merge(d[cols], on=['permno', 'dlycaldt'], how='left')


def period_end_timestamps_from_calendar(all_dates, trade_freq):
    """在样本内日历上取各周/月/季的最后一个交易日（对齐作者按周期末取样，而非逐日滚动）。

    说明：作者用 SPY 的周期末日列表；无 CACHE 时用 CRSP 样本中出现的日期构造周期末，
    主流实证中与 SPY 日历高度重合，极端情况下可能差 1 个交易日。
    """
    uniq = pd.Series(pd.to_datetime(all_dates).unique()).sort_values()
    ddf = pd.DataFrame({'d': uniq})
    if trade_freq == 'week':
        ddf['p'] = ddf['d'].dt.to_period('W-FRI')
    elif trade_freq == 'month':
        ddf['p'] = ddf['d'].dt.to_period('M')
    elif trade_freq == 'quarter':
        ddf['p'] = ddf['d'].dt.to_period('Q')
    else:
        raise ValueError(f"trade_freq must be week|month|quarter, got {trade_freq}")
    ends = ddf.groupby('p', sort=True)['d'].max()
    return set(pd.Timestamp(x).normalize() for x in ends.values)

def set_seed(seed):
    """全局随机种子，确保论文复现严谨性"""
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)
    if hasattr(torch.backends, "mps") and torch.backends.mps.is_available():
        torch.mps.manual_seed(seed)

# --- 动态数据集 (防止内存崩溃的核心) ---
class StockImageDataset(Dataset):
    def __init__(
        self,
        data_df,
        F_horizon,
        L_window,
        mu_train=None,
        std_train=None,
        is_train=False,
        trade_freq=None,
        period_end_set=None,
    ):
        self.L = L_window
        self.cfg = MODEL_CFG[L_window]
        self.F = F_horizon
        self.mu = mu_train
        self.std = std_train
        self.is_train = is_train
        if trade_freq is None:
            trade_freq = PREDICT_HORIZON_TO_TRADE_FREQ.get(F_horizon)
        if trade_freq is None:
            raise ValueError(f"无法推断 trade_freq，请显式传入（F_horizon={F_horizon}）")
        self.trade_freq = trade_freq

        # 重置索引，确保 g.index.values 与后续 numpy array 的 0-based 索引一致防越界
        data_df = data_df.reset_index(drop=True)

        # 将原始数据转化为 numpy 方便索引提速十倍
        self.P_adj = data_df[['dlyopen_adj', 'dlyhigh_adj', 'dlylow_adj', 'dlyclose_adj']].values
        self.V = data_df['dlyvol'].values
        self.labels = data_df[f'Label_{F_horizon}'].values
        self._dates = pd.to_datetime(data_df['dlycaldt']).values

        if period_end_set is not None:
            self._period_end_set = period_end_set
        else:
            self._period_end_set = period_end_timestamps_from_calendar(self._dates, trade_freq)

        self.valid_indices = []
        grp = data_df.groupby('permno', group_keys=False)
        for perm, g in grp:
            idxs = g.index.values
            # 移动平均线需要额外的 L - 1 天历史数据，所以起始 idx 必须 >= 2L - 1
            if len(idxs) < 2 * self.L - 1 + self.F:
                continue
            for t_idx in idxs[2 * self.L - 1 : len(idxs) - self.F]:
                ts = pd.Timestamp(self._dates[t_idx]).normalize()
                if ts in self._period_end_set:
                    self.valid_indices.append(int(t_idx))
                
    def __len__(self):
        return len(self.valid_indices)

    def __getitem__(self, idx):
        t_idx = self.valid_indices[idx]
        
        # P_full 提取 2*L - 1 的数据以保证第一天有它的完整 L 周期 SMA，这里是 [t_idx - 2L + 1, t_idx)
        P_full = self.P_adj[t_idx - 2 * self.L + 1 : t_idx]
        V_win = self.V[t_idx - self.L : t_idx]
        label = self.labels[t_idx]
        
        P_win = P_full[-self.L:]
        V_min, V_max = V_win.min(), V_win.max()
        V_range = V_max - V_min + 1e-8
        
        # --- 移动平均线计算 ---
        sma_vals = np.zeros(self.L)
        for tau in range(self.L):
            sma_vals[tau] = P_full[tau : tau + self.L, 3].mean()
            
        # P_min 和 P_max 需要将 SMA 的值纳入比对池中
        P_min = min(P_win.min(), sma_vals.min())
        P_max = max(P_win.max(), sma_vals.max())
        P_range = P_max - P_min + 1e-8
        
        ohlc_h = self.cfg['H_ohlc']
        vol_h = self.cfg['H_vol']
        total_w = self.cfg['W']
        
        def ret_to_yaxis(ret):
            pixels_per_unit = (ohlc_h - 1.0) / P_range
            return int(np.around((ret - P_min) * pixels_per_unit))
            
        # --- 精确复现其原版 PIL `ImageDraw` 绘图 ---
        ohlc_img = Image.new("L", (total_w, ohlc_h), 0)
        draw_ohlc = ImageDraw.Draw(ohlc_img)
        pixels_ohlc = ohlc_img.load()
        
        # 1. 画 SMA 连线 (原版：用 draw.line 直接连线即可，且宽度1px，色值255)
        for tau in range(self.L - 1):
            x_start = tau * 3 + 1
            x_end = (tau + 1) * 3 + 1
            y_start = ret_to_yaxis(sma_vals[tau])
            y_end = ret_to_yaxis(sma_vals[tau+1])
            draw_ohlc.line((x_start, y_start, x_end, y_end), width=1, fill=255)
            
        # 2. 画 OHLC 线
        for tau in range(self.L):
            x_c = tau * 3 + 1
            y_high = ret_to_yaxis(P_win[tau, 1])
            y_low = ret_to_yaxis(P_win[tau, 2])
            y_open = ret_to_yaxis(P_win[tau, 0])
            y_close = ret_to_yaxis(P_win[tau, 3])
            
            # High to Low
            for j in range(min(y_low, y_high), max(y_low, y_high) + 1):
                pixels_ohlc[x_c, j] = 255
            
            # Open (x_c-1 to x_c 之间)
            for i in range(x_c - 1, x_c + 1):
                pixels_ohlc[i, y_open] = 255
                
            # Close (x_c+1)
            pixels_ohlc[x_c + 1, y_close] = 255
            
        # 根据原版代码库逻辑：OHLC 图是在生成后进行 FLIP_TOP_BOTTOM 上下翻转
        ohlc_img = ohlc_img.transpose(Image.FLIP_TOP_BOTTOM)
        
        # 3. 画 Volume (放在倒数的区域里，这里我们也跟原版对齐，原版直接是长条往下画)
        vol_img = Image.new("L", (total_w, vol_h), 0)
        pixels_vol = vol_img.load()
        
        if (not np.isnan(V_max)) and V_max != 0:
            pixels_per_volume = 1.0 * vol_h / abs(V_max)
            for tau in range(self.L):
                if np.isnan(V_win[tau]): 
                    continue
                v_h_curr = int(np.around(abs(V_win[tau]) * pixels_per_volume))
                x_c = tau * 3 + 1
                # 因为在 PIL 里 Y=0 在顶端，需要从底边 (vol_h-1) 开始填补，不过原图做了 FlipTopToBottom
                # 此处保持与原代码严格一致，它对 Volume 的画法：pixels[int(centers[day]), vol_height - 1] = 255 但后续由于没 Flip，我们将其调整：
                v_h_curr = max(1, min(vol_h, v_h_curr)) # 保证至少有一个像素不越界
                for j in range(vol_h - v_h_curr, vol_h):
                    pixels_vol[x_c, j] = 255
                    
        # 组装 Image
        full_img = Image.new("L", (total_w, self.cfg['H']), 0)
        full_img.paste(ohlc_img, (0, 0))
        # GAP 是 1
        full_img.paste(vol_img, (0, ohlc_h + 1))
        
        # 转回 array
        img = np.array(full_img, dtype=np.float32) / 255.0
        
        if self.mu is not None and self.std is not None:
            img = (img - self.mu) / (self.std + 1e-8)
            
        return torch.tensor(img, dtype=torch.float32).unsqueeze(0), torch.tensor(label, dtype=torch.long)


# --- 定义模型 (100%匹配 README 表格，使用 Xavier, 50% Dropout) ---
def init_weights(m):
    """Xavier 均匀初始化"""
    if isinstance(m, nn.Conv2d) or isinstance(m, nn.Linear):
        nn.init.xavier_uniform_(m.weight)
        if m.bias is not None:
            nn.init.zeros_(m.bias)

class I5Model(nn.Module):
    def __init__(self):
        super(I5Model, self).__init__()
        self.block1 = nn.Sequential(
            nn.Conv2d(1, 64, kernel_size=(5, 3), stride=(1, 1), padding='same', bias=True),
            nn.BatchNorm2d(64),
            nn.LeakyReLU(0.01),
            nn.MaxPool2d(kernel_size=(2, 1), stride=(2, 1))
        )
        self.block2 = nn.Sequential(
            nn.Conv2d(64, 128, kernel_size=(5, 3), stride=(1, 1), padding='same', bias=True),
            nn.BatchNorm2d(128),
            nn.LeakyReLU(0.01),
            nn.MaxPool2d(kernel_size=(2, 1), stride=(2, 1))
        )
        self.fc = nn.Sequential(
            nn.Flatten(),
            nn.Dropout(0.5),
            nn.Linear(15360, 2), # 128 * (32//2//2=8) * (15)= 15360
        )
    def forward(self, x):
        x = self.block1(x)
        x = self.block2(x)
        return self.fc(x)

class I20Model(nn.Module):
    def __init__(self):
        super(I20Model, self).__init__()
        self.block1 = nn.Sequential(
            nn.Conv2d(1, 64, kernel_size=(5, 3), stride=(3, 1), dilation=(2, 1), padding=(2, 1), bias=True),
            nn.BatchNorm2d(64),
            nn.LeakyReLU(0.01),
            nn.MaxPool2d(kernel_size=(2, 1), stride=(2, 1), ceil_mode=True)
        )
        self.block2 = nn.Sequential(
            nn.Conv2d(64, 128, kernel_size=(5, 3), stride=(1, 1), dilation=(1, 1), padding=(2, 1), bias=True),
            nn.BatchNorm2d(128),
            nn.LeakyReLU(0.01),
            nn.MaxPool2d(kernel_size=(2, 1), stride=(2, 1), ceil_mode=True)
        )
        self.block3 = nn.Sequential(
            nn.Conv2d(128, 256, kernel_size=(5, 3), stride=(1, 1), dilation=(1, 1), padding=(2, 1), bias=True),
            nn.BatchNorm2d(256),
            nn.LeakyReLU(0.01),
            nn.MaxPool2d(kernel_size=(2, 1), stride=(2, 1), ceil_mode=True)
        )
        self.fc = nn.Sequential(
            nn.Flatten(),
            nn.Dropout(0.5),
            nn.Linear(46080, 2), # 256 * 3 * 60 = 46080
        )
    def forward(self, x):
        x = self.block1(x)
        x = self.block2(x)
        x = self.block3(x)
        return self.fc(x)

class I60Model(nn.Module):
    def __init__(self):
        super(I60Model, self).__init__()
        self.block1 = nn.Sequential(
            nn.Conv2d(1, 64, kernel_size=(5, 3), stride=(3, 1), dilation=(3, 1), padding=(2, 1), bias=True),
            nn.BatchNorm2d(64),
            nn.LeakyReLU(0.01),
            nn.MaxPool2d(kernel_size=(2, 1), stride=(2, 1), ceil_mode=True)
        )
        self.block2 = nn.Sequential(
            nn.Conv2d(64, 128, kernel_size=(5, 3), stride=(1, 1), dilation=(1, 1), padding=(2, 1), bias=True),
            nn.BatchNorm2d(128),
            nn.LeakyReLU(0.01),
            nn.MaxPool2d(kernel_size=(2, 1), stride=(2, 1), ceil_mode=True)
        )
        self.block3 = nn.Sequential(
            nn.Conv2d(128, 256, kernel_size=(5, 3), stride=(1, 1), dilation=(1, 1), padding=(2, 1), bias=True),
            nn.BatchNorm2d(256),
            nn.LeakyReLU(0.01),
            nn.MaxPool2d(kernel_size=(2, 1), stride=(2, 1), ceil_mode=True)
        )
        self.block4 = nn.Sequential(
            nn.Conv2d(256, 512, kernel_size=(5, 3), stride=(1, 1), dilation=(1, 1), padding=(2, 1), bias=True),
            nn.BatchNorm2d(512),
            nn.LeakyReLU(0.01),
            nn.MaxPool2d(kernel_size=(2, 1), stride=(2, 1), ceil_mode=True)
        )
        self.fc = nn.Sequential(
            nn.Flatten(),
            nn.Dropout(0.5),
            nn.Linear(184320, 2), # 512 * 2 * 180 = 184320
        )
    def forward(self, x):
        x = self.block1(x)
        x = self.block2(x)
        x = self.block3(x)
        x = self.block4(x)
        return self.fc(x)

import matplotlib.pyplot as plt

def display_dynamic_image(L, F, title):
    # 遍历股票，直到找到一只拥有足够长历史数据来生成该窗口图像的股票
    for perm in df_clean['permno'].unique():
        demo_df = df_clean[df_clean['permno'] == perm].copy()
        if len(demo_df) < 2 * L - 1 + F:
            continue
            
        ds_demo = StockImageDataset(demo_df, F_horizon=F, L_window=L)
        if len(ds_demo) > 0:
            img_tensor, lbl = ds_demo[0]
            img = img_tensor.squeeze(0).numpy()
            plt.figure(figsize=(img.shape[1]/5, img.shape[0]/5)) 
            plt.imshow(img, cmap='gray', origin='upper')
            plt.title(f"{title} - Label: {lbl}", fontsize=12)
            plt.axis('off')
            plt.show()
            break  # 画出一张后就退出

# 验证：生成 5、20、60天的缩略图
display_dynamic_image(5, 5, "I5 CNN Image (H=32 x W=15)")
display_dynamic_image(20, 20, "I20 CNN Image (H=64 x W=60)")
display_dynamic_image(60, 60, "I60 CNN Image (H=96 x W=180)")

print("== 基础防爆和 5天、20天、60天模型架构设置完成，执行即可 ==")

## 四、数据集划分与 CNN 训练

`split_and_prepare_datasets`：训练/验证在样本级别随机 70/30 切分（与作者 ConcatDataset + random_split 一致）、标准化参数缓存；对多个 horizon 循环训练并保存权重。

**依赖**：`df_clean`、`device`、`MODEL_DIR`。

In [ ]:
# ==========================================
# 步骤 4 ~ 6：数据集时序划分、标准化提取与单种子强力训练大循环
# ==========================================
import copy
from torch.utils.data import DataLoader, random_split


def _training_seeds():
    return list(SMOKE_SEEDS) if globals().get("SMOKE_TEST", False) else [42, 123, 456, 789, 101112]


def has_complete_checkpoints(L=5):
    seeds = _training_seeds()
    ckpt_paths = [os.path.join(MODEL_DIR, f"I{L}_checkpoint{seed}.pth.tar") for seed in seeds]
    return all(os.path.exists(p) for p in ckpt_paths)


def split_and_prepare_datasets(df, L=5, F=5, build_train_val=True):
    """
    严格按照论文进行数据集划分：
    - Train+Val: 1993 ~ 2000
    - Test: 2001 ~ 2019
    - Train/Val 之间在样本级别随机 7:3 切分（与作者 ConcatDataset + random_split 一致）

    SMOKE_TEST=True 时：在当前子样本内按时间切出 Train+Val 与 Test，
    再在 Train+Val 段内做样本级 random_split 7:3。
    """
    print(f"\n====== 划分数据集 (L={L}, F={F}) ======")

    if globals().get("SMOKE_TEST", False):
        u = np.sort(df["dlycaldt"].unique())
        if len(u) < 40:
            raise ValueError("SMOKE: 交易日过少，请放宽 SMOKE_DATE_START/END 或增大 SMOKE_READ_CSV_NROWS")
        split_idx = min(int(len(u) * 0.78), len(u) - 12)
        split_idx = max(split_idx, max(12, int(len(u) * 0.55)))
        split_date = pd.Timestamp(u[split_idx])
        df_tv = df[df["dlycaldt"] <= split_date].copy()
        df_test = df[df["dlycaldt"] > split_date].copy()
        print(f"[SMOKE] 训练+验证窗口 ≤ {split_date.date()} | 测试窗口 > {split_date.date()}")
    else:
        df_tv = df[(df["dlycaldt"] >= "1993-01-01") & (df["dlycaldt"] <= "2000-12-31")].copy()
        df_test = df[(df["dlycaldt"] >= "2001-01-01") & (df["dlycaldt"] <= "2019-12-31")].copy()

    print(f"数据集行数: Train+Val={len(df_tv)}, Test={len(df_test)}")

    trade_freq = PREDICT_HORIZON_TO_TRADE_FREQ.get(F)
    if trade_freq is None:
        raise ValueError(f"不支持的预测 horizon F={F}，应为 5/20/60")
    print(f"训练样本频率（与作者一致）: horizon F={F} -> trade_freq={trade_freq}（仅周期末日取样）")

    cal_dates = pd.concat([df_tv, df_test], axis=0)["dlycaldt"]
    period_end_set = period_end_timestamps_from_calendar(cal_dates, trade_freq)

    # ---- mean/std 估计（与作者一致：用全部 IS 数据的前 50000 样本） ----
    smoke_tag = "_smoke" if globals().get("SMOKE_TEST", False) else ""
    norm_cache_path = os.path.join(MODEL_DIR, f"mean_std_I{L}_R{F}_train{smoke_tag}.npz")
    if os.path.exists(norm_cache_path):
        cache = np.load(norm_cache_path)
        mu_train = float(cache["mean"])
        std_train = float(cache["std"])
        print(f"读取缓存标准化参数: μ={mu_train:.6f}, σ={std_train:.6f}")
    else:
        ds_tv_raw = StockImageDataset(df_tv, F, L, is_train=True, trade_freq=trade_freq, period_end_set=period_end_set)
        cap = int(SMOKE_NORM_SAMPLES) if globals().get("SMOKE_TEST", False) else 50000
        print(f"计算 IS 全局 μ 和 σ (L={L}, F={F}, 最多 {cap} 张图) ...")
        sample_n = min(cap, len(ds_tv_raw))
        pixel_sum = 0.0
        pixel_sq_sum = 0.0
        pixel_count = 0

        for i in range(sample_n):
            img, _ = ds_tv_raw[i]
            arr = img.numpy()
            pixel_sum += float(arr.sum())
            pixel_sq_sum += float((arr * arr).sum())
            pixel_count += arr.size

        mu_train = pixel_sum / max(pixel_count, 1)
        var_train = pixel_sq_sum / max(pixel_count, 1) - mu_train * mu_train
        std_train = float(np.sqrt(max(var_train, 1e-12)))

        np.savez(norm_cache_path, mean=mu_train, std=std_train)
        print(f"已缓存标准化参数到 {norm_cache_path}")
        print(f"==> 归一化参数：μ={mu_train:.6f}, σ={std_train:.6f}")

    # ---- 构建带标准化的数据集 ----
    if build_train_val:
        ds_tv = StockImageDataset(
            df_tv, F, L,
            mu_train=mu_train, std_train=std_train,
            is_train=True, trade_freq=trade_freq, period_end_set=period_end_set,
        )
        train_size = int(len(ds_tv) * 0.7)
        val_size = len(ds_tv) - train_size
        print(f"样本级 random_split: Train={train_size}, Val={val_size} (总 {len(ds_tv)})")

        generator = torch.Generator().manual_seed(42)
        ds_train, ds_val = random_split(ds_tv, [train_size, val_size], generator=generator)
    else:
        print(f"检测到 I{L} checkpoint 完整，跳过 Train/Val 构建与 random_split")
        ds_train, ds_val = None, None

    del df_tv

    ds_test = StockImageDataset(df_test, F, L, mu_train=mu_train, std_train=std_train, trade_freq=trade_freq, period_end_set=period_end_set)

    return ds_train, ds_val, ds_test, df_test


def train_single_seed_model(ds_train, ds_val, L=5):
    """按种子训练；SMOKE 模式下极少 epoch、单种子。"""
    SEEDS = _training_seeds()
    BATCH_SIZE = int(SMOKE_BATCH_SIZE) if globals().get("SMOKE_TEST", False) else 128
    PATIENCE = 1 if globals().get("SMOKE_TEST", False) else 2
    MAX_EPOCHS = int(SMOKE_MAX_EPOCHS) if globals().get("SMOKE_TEST", False) else 50

    train_loader = make_dataloader(ds_train, batch_size=BATCH_SIZE, shuffle=True)
    val_loader = make_dataloader(ds_val, batch_size=BATCH_SIZE, shuffle=False)

    models = []

    for seed in SEEDS:
        print(f"\n>>>> 开始训练模型 L={L} [Seed={seed}] <<<<")
        set_seed(seed)

        if L == 5:
            model = I5Model().to(device)
        elif L == 20:
            model = I20Model().to(device)
        elif L == 60:
            model = I60Model().to(device)
        else:
            raise NotImplementedError(f"未配置支持的 L={L}")

        model.apply(init_weights)
        model = maybe_compile(model)

        criterion = nn.CrossEntropyLoss()
        optimizer = optim.Adam(model.parameters(), lr=1e-5, betas=(0.9, 0.999), eps=1e-8)

        best_val_loss = float("inf")
        best_val_acc = 0.0
        best_epoch = 0
        best_weights = copy.deepcopy(model.state_dict())
        patience_counter = 0

        for epoch in range(1, MAX_EPOCHS + 1):
            model.train()
            train_loss = 0.0

            pbar = tqdm(train_loader, desc=f"Epoch {epoch} Train (L={L})", leave=False)
            for X_batch, y_batch in pbar:
                X_batch, y_batch = X_batch.to(device, non_blocking=PIN_MEMORY), y_batch.to(device, non_blocking=PIN_MEMORY)

                optimizer.zero_grad()
                outputs = model(X_batch)
                loss = criterion(outputs, y_batch)
                loss.backward()
                optimizer.step()

                train_loss += loss.item() * X_batch.size(0)
                pbar.set_postfix({"loss": f"{loss.item():.4f}"})

            train_loss /= max(len(ds_train), 1)

            model.eval()
            val_loss = 0.0
            val_correct = 0
            with torch.no_grad():
                for X_batch, y_batch in val_loader:
                    X_batch, y_batch = X_batch.to(device, non_blocking=PIN_MEMORY), y_batch.to(device, non_blocking=PIN_MEMORY)
                    outputs = model(X_batch)
                    loss = criterion(outputs, y_batch)
                    val_loss += loss.item() * X_batch.size(0)
                    preds = outputs.argmax(dim=1)
                    val_correct += (preds == y_batch).sum().item()

            val_loss /= max(len(ds_val), 1)
            val_acc = val_correct / max(len(ds_val), 1)

            print(f"L={L} Epoch {epoch} | Train Loss={train_loss:.4f} | Val Loss={val_loss:.4f} | Val Acc={val_acc:.4f}")

            if val_loss < best_val_loss:
                best_val_loss = val_loss
                best_val_acc = val_acc
                best_epoch = epoch
                best_weights = copy.deepcopy(model.state_dict())
                patience_counter = 0
            else:
                patience_counter += 1

            if patience_counter >= PATIENCE:
                print(f"Early Stopping 触发！针对 L={L} 最终选用 Val Loss: {best_val_loss:.4f}")
                break

        model.load_state_dict(best_weights)
        models.append(model)

        checkpoint = {
            "loss": float(best_val_loss),
            "accy": float(best_val_acc),
            "epoch": int(best_epoch),
            "seed": int(seed),
            "model_state_dict": copy.deepcopy(best_weights),
        }
        ckpt_path = os.path.join(MODEL_DIR, f"I{L}_checkpoint{seed}.pth.tar")
        torch.save(checkpoint, ckpt_path)

    return models


def load_or_train_models(ds_train, ds_val, L=5):
    """优先加载已有 checkpoint；SMOKE 模式下仅检查当前种子列表对应的文件。"""
    seeds = _training_seeds()
    ckpt_paths = [os.path.join(MODEL_DIR, f"I{L}_checkpoint{seed}.pth.tar") for seed in seeds]

    if L == 5:
        model_cls = I5Model
    elif L == 20:
        model_cls = I20Model
    elif L == 60:
        model_cls = I60Model
    else:
        raise NotImplementedError(f"未配置支持的 L={L}")

    if all(os.path.exists(p) for p in ckpt_paths):
        print(f"\n>>>> 检测到 I{L} 的 {len(ckpt_paths)} 个 checkpoint，直接加载免训练 <<<<")
        models = []
        for p in ckpt_paths:
            checkpoint = torch.load(p, map_location=device)
            model = model_cls().to(device)
            sd = checkpoint["model_state_dict"]
            # torch.compile 保存的 checkpoint 带 _orig_mod. 前缀，需要去掉
            if any(k.startswith("_orig_mod.") for k in sd.keys()):
                sd = {k.removeprefix("_orig_mod."): v for k, v in sd.items()}
            model.load_state_dict(sd)
            model.eval()
            model = maybe_compile(model)
            models.append(model)
        return models
    print(f"\n>>>> I{L} checkpoint 不完整，降级回退至训练模式 <<<<")
    if ds_train is None or ds_val is None:
        raise ValueError(f"I{L} checkpoint 不完整，但未提供训练集/验证集，无法回退训练")
    return train_single_seed_model(ds_train, ds_val, L=L)


# ==================================
# 按 SMOKE_TASKS 构建与训练（全量时为 I5/I20/I60）
# ==================================
_g = globals()
for L, F in SMOKE_TASKS:
    assert L == F, "当前 notebook 假设 L 与 F 一致（I5/I20/I60）"
    ckpt_ready = has_complete_checkpoints(L)
    ds_tr, ds_va, ds_te, df_te = split_and_prepare_datasets(df_clean, L=L, F=F, build_train_val=(not ckpt_ready))
    mdls = load_or_train_models(ds_tr, ds_va, L=L)
    _g[f"ds_train_{L}"] = ds_tr
    _g[f"ds_val_{L}"] = ds_va
    _g[f"ds_test_{L}"] = ds_te
    _g[f"df_test_{L}"] = df_te
    _g[f"models_{L}"] = mdls


## 五、测试集集成预测与多空回测

集成预测 `predict_ensemble`、`evaluate_strategy` 与换手惩罚后的夏普；生成各 horizon 的测试面板与收益序列供后续图表使用。

In [ ]:
# ==========================================
# 步骤 7 & 8：测试集全量外推预测、多空组合构建与扣费/不扣费双版本夏普回测 (2001~2019)
# ==========================================
import time
import matplotlib.pyplot as plt

def predict_ensemble(models, ds_test, batch_size=None):
    """提取集成模型打分：通过前向推理计算测试集的 softmax 正类概率平均值"""
    # 必须用局部绑定：训练格里的 `for L, F in SMOKE_TASKS` 会把全局 `F` 覆盖成 int，勿写 F.softmax
    _softmax = torch.nn.functional.softmax
    batch_size = int(PRED_BATCH_SIZE if batch_size is None else batch_size)
    use_amp = bool(USE_FP16_INFER and device.type == "cuda")
    print(f"====== 开始运行测试集集成预测 ====== (batch_size={batch_size}, amp={use_amp})")
    test_loader = make_dataloader(ds_test, batch_size=batch_size, shuffle=False)

    for m in models:
        m.eval()

    all_preds = []
    total_batches = len(test_loader)
    heartbeat_every = max(1, total_batches // 50)
    t0 = time.time()

    with torch.inference_mode():
        for bi, (X_batch, _) in enumerate(tqdm(test_loader, desc="Ensemble Predicting"), start=1):
            X_batch = X_batch.to(device, non_blocking=PIN_MEMORY)
            # 初始化累加器
            batch_probs = torch.zeros(X_batch.size(0), device=device)
            for m in models:
                with torch.autocast(device_type="cuda", dtype=torch.float16, enabled=use_amp):
                    outputs = m(X_batch)
                probs = _softmax(outputs, dim=1)[:, 1]
                batch_probs += probs

            # 集成平均
            batch_probs /= len(models)
            all_preds.extend(batch_probs.float().cpu().numpy())

            # papermill 对 tqdm 刷新不稳定，增加心跳日志便于实时监控
            if bi == 1 or bi % heartbeat_every == 0 or bi == total_batches:
                elapsed = time.time() - t0
                print(
                    f"[Ensemble Progress] {bi}/{total_batches} batches "
                    f"({bi / max(total_batches, 1) * 100:.1f}%) | "
                    f"preds={len(all_preds)} | elapsed={elapsed:.1f}s"
                )

    return np.array(all_preds)

def evaluate_strategy(df_test_valid, F_horizon=5, fee_rate=0.001, df_daily=None):
    """
    极速量化回测 (最终极强修复版)：
    1. 调仓日与训练一致：仅为周期末日（周/月/季），不再用「每 F 个交易日」近似。
    2. Baseline 动量/均线在日频面板上计算后 merge 到周期末截面（避免稀疏行 pct_change 错位）。
    3. EW / VW 多空；年化夏普与作者一致采用每年 52/12/4 期。
    """
    print(f"\n====== 开始全面量化回测评估 L={F_horizon} (包含 VW市值加权 与 多重基础因子) ======")

    df_test_valid = df_test_valid.copy()

    if df_daily is not None:
        df_test_valid = enrich_rebalance_panel_with_daily_baselines(df_test_valid, df_daily, F_horizon)
    else:
        df_test_valid['mom_score'] = df_test_valid.groupby('permno')['dlyclose_adj'].pct_change(F_horizon).fillna(0)
        ma_fast = df_test_valid.groupby('permno')['dlyclose_adj'].transform(
            lambda x: x.rolling(max(2, F_horizon // 2), min_periods=1).mean()
        )
        ma_slow = df_test_valid.groupby('permno')['dlyclose_adj'].transform(
            lambda x: x.rolling(max(20, F_horizon), min_periods=1).mean()
        )
        df_test_valid['ma_score'] = (ma_fast / ma_slow - 1).fillna(0)
    
    # 为策略与各种 Baseline 打分/分十等份
    df_test_valid['rank_pred'] = df_test_valid.groupby('dlycaldt')['pred_score'].rank(pct=True)
    df_test_valid['rank_mom'] = df_test_valid.groupby('dlycaldt')['mom_score'].rank(pct=True)
    df_test_valid['rank_ma'] = df_test_valid.groupby('dlycaldt')['ma_score'].rank(pct=True)
    
    # 对各个子策略生成 Port_xxx 标签
    for strat in ['pred', 'mom', 'ma']:
        col_name = f'Port_{strat}'
        df_test_valid[col_name] = 'Others'
        df_test_valid.loc[df_test_valid[f'rank_{strat}'] >= 0.90, col_name] = 'D10 (Long)'
        df_test_valid.loc[df_test_valid[f'rank_{strat}'] <= 0.10, col_name] = 'D1 (Short)'
    
    # 调仓日：预测面板已为周期末日（与 StockImageDataset.valid_indices 一致），每个日期一次调仓
    df_reb = df_test_valid.copy()
    
    # ================= [新补齐] 定制化截面加权收益推导函数 =================
    def calc_portfolio_return(df_group, target_col='Port_pred', weight_mode='EW'):
        # 对于指定的组合策略和截面调盘日进行收益率计算
        if weight_mode == 'EW':
            res = df_group.groupby(['dlycaldt', target_col])[f'R_fut_{F_horizon}'].mean().unstack().fillna(0)
        else: # VW
            df_group['g_cap_sum'] = df_group.groupby(['dlycaldt', target_col])['dlycap'].transform('sum')
            # 根据前一天或当天结算市值作为资金分配比重
            df_group['vw_ret'] = df_group[f'R_fut_{F_horizon}'] * (df_group['dlycap'] / (df_group['g_cap_sum'] + 1e-8))
            res = df_group.groupby(['dlycaldt', target_col])['vw_ret'].sum().unstack().fillna(0)
            
        return res.get('D10 (Long)', pd.Series(0, index=res.index)), res.get('D1 (Short)', pd.Series(0, index=res.index))
    
    # 生成基础记录表
    period_ret = pd.DataFrame(index=np.sort(df_reb['dlycaldt'].unique()))
    
    # 提取多空表现
    cnn_d10_ew, cnn_d1_ew = calc_portfolio_return(df_reb, 'Port_pred', 'EW')
    cnn_d10_vw, cnn_d1_vw = calc_portfolio_return(df_reb, 'Port_pred', 'VW') # 市值加权
    mom_d10_ew, mom_d1_ew = calc_portfolio_return(df_reb, 'Port_mom', 'EW')
    ma_d10_ew, ma_d1_ew = calc_portfolio_return(df_reb, 'Port_ma', 'EW')
    
    # 多空毛收益率
    period_ret['CNN_LS_gross_EW'] = cnn_d10_ew - cnn_d1_ew
    period_ret['CNN_LS_gross_VW'] = cnn_d10_vw - cnn_d1_vw
    period_ret['Mom_LS_gross'] = mom_d10_ew - mom_d1_ew
    period_ret['MA_LS_gross'] = ma_d10_ew - ma_d1_ew
    
    # 多空净收益率（扣减交易成本；fee_rate 为单边成本，默认 0.001=10bps）
    period_ret['CNN_LS_net_EW'] = period_ret['CNN_LS_gross_EW'] - (fee_rate * 2)
    period_ret['CNN_LS_net_VW'] = period_ret['CNN_LS_gross_VW'] - (fee_rate * 2)
    period_ret['Mom_LS_net'] = period_ret['Mom_LS_gross'] - (fee_rate * 2)
    period_ret['MA_LS_net'] = period_ret['MA_LS_gross'] - (fee_rate * 2)
    
    # 保证后续因子回归脚本不出错，赋值一个默认净值
    period_ret['CNN_LS_net'] = period_ret['CNN_LS_net_EW'] 

    # ============== 夏普核算与表现打印（作者：52/12/4 期每年）==============
    ppy = PERIODS_PER_YEAR[F_horizon]

    def eval_metrics(series, name, version):
        m = series.mean()
        s = series.std()
        sharpe = (m / s) * np.sqrt(ppy) if s > 0 else 0
        print(f"【{name}｜{version}】 年化收益: {m * ppy * 100:>6.2f}% | 夏普比率: {sharpe:.4f}")

    print("\n>>>>>>> [外推期 2001-2019] 回测核心绩效表（Gross：不扣交易成本） <<<<<<<")
    eval_metrics(period_ret['Mom_LS_gross'], "传统动量 Baseline EW", "Gross")
    eval_metrics(period_ret['MA_LS_gross'], "移动均线交叉 MA EW", "Gross")
    eval_metrics(period_ret['CNN_LS_gross_EW'], "CNN 图像等权策略 EW", "Gross")
    eval_metrics(period_ret['CNN_LS_gross_VW'], "CNN 图像市值权策略 VW", "Gross")

    print(f"\n>>>>>>> [外推期 2001-2019] 回测核心绩效表（Net：扣除单边 fee_rate={fee_rate:.4%}，多空每期合计扣 {fee_rate * 2:.4%}） <<<<<<<")
    eval_metrics(period_ret['Mom_LS_net'], "传统动量 Baseline EW", "Net")
    eval_metrics(period_ret['MA_LS_net'], "移动均线交叉 MA EW", "Net")
    eval_metrics(period_ret['CNN_LS_net_EW'], "CNN 图像等权策略 EW", "Net")
    eval_metrics(period_ret['CNN_LS_net_VW'], "CNN 图像市值权策略 VW", "Net")

    # =============== 累计对数收益绘图（净收益：含交易成本）=================
    plt.figure(figsize=(14, 7))
    plt.plot(period_ret.index, np.log(1 + period_ret['CNN_LS_net_EW']).cumsum(), label=f'CNN EW Long-Short (Net)', color='#c0392b', linewidth=3)
    plt.plot(period_ret.index, np.log(1 + period_ret['CNN_LS_net_VW']).cumsum(), label=f'CNN VW Long-Short (Net)', color='#e74c3c', linewidth=2, linestyle='-.')
    plt.plot(period_ret.index, np.log(1 + period_ret['Mom_LS_net']).cumsum(), label=f'Baseline Momentum (Net)', color='#7f8c8d', linewidth=2, linestyle='--')
    plt.plot(period_ret.index, np.log(1 + period_ret['MA_LS_net']).cumsum(), label=f'Baseline MA Cross (Net)', color='#34495e', linewidth=2, linestyle=':')
    plt.title(f'Cumulative Log Returns — Net of Costs (L={F_horizon}, round-trip levy = {fee_rate*2})', fontsize=14, fontweight='bold')
    plt.xlabel('Date (Year)', fontsize=13)
    plt.ylabel('Cumulative Log Return', fontsize=13)
    plt.legend(loc='upper left', fontsize=11, frameon=True, edgecolor='black')
    plt.grid(True, which='both', alpha=0.2, linestyle='-')
    plt.tight_layout()
    plt.show()

    # =============== 累计对数收益绘图（毛收益：不计交易成本）=================
    plt.figure(figsize=(14, 7))
    plt.plot(period_ret.index, np.log(1 + period_ret['CNN_LS_gross_EW']).cumsum(), label='CNN EW Long-Short (Gross)', color='#c0392b', linewidth=3)
    plt.plot(period_ret.index, np.log(1 + period_ret['CNN_LS_gross_VW']).cumsum(), label='CNN VW Long-Short (Gross)', color='#e74c3c', linewidth=2, linestyle='-.')
    plt.plot(period_ret.index, np.log(1 + period_ret['Mom_LS_gross']).cumsum(), label='Baseline Momentum (Gross)', color='#7f8c8d', linewidth=2, linestyle='--')
    plt.plot(period_ret.index, np.log(1 + period_ret['MA_LS_gross']).cumsum(), label='Baseline MA Cross (Gross)', color='#34495e', linewidth=2, linestyle=':')
    plt.title(f'Cumulative Log Returns — Gross of Costs (L={F_horizon}, no fee deduction)', fontsize=14, fontweight='bold')
    plt.xlabel('Date (Year)', fontsize=13)
    plt.ylabel('Cumulative Log Return', fontsize=13)
    plt.legend(loc='upper left', fontsize=11, frameon=True, edgecolor='black')
    plt.grid(True, which='both', alpha=0.2, linestyle='-')
    plt.tight_layout()
    plt.show()
    
    return period_ret

# ==========================================
# 完整运行触发控制台：获取各个周期的回测表现统计
# ==========================================
print("\n============ 启动所有预测及回测流程 ============\n")

# ---------- 回测磁盘缓存辅助 ----------
def _bt_cache_path(horizon):
    smoke_tag = "_smoke" if SMOKE_TEST else ""
    return os.path.join(CACHE_DIR, f"backtest_I{horizon}{smoke_tag}.pkl")

def _load_bt_cache(horizon):
    """尝试从磁盘加载回测结果，成功返回 DataFrame，否则返回 None。"""
    p = _bt_cache_path(horizon)
    if os.path.exists(p):
        print(f"💾 [磁盘缓存命中] 加载 backtest_I{horizon} ← {p}")
        return pd.read_pickle(p)
    return None

def _save_bt_cache(df, horizon):
    """将回测结果保存到磁盘。"""
    p = _bt_cache_path(horizon)
    df.to_pickle(p)
    print(f"💾 [回测已缓存] backtest_I{horizon} → {p}")


def _pred_cache_path(horizon):
    smoke_tag = "_smoke" if SMOKE_TEST else ""
    return os.path.join(CACHE_DIR, f"preds_I{horizon}{smoke_tag}.npy")


def _load_pred_cache(horizon):
    p = _pred_cache_path(horizon)
    if os.path.exists(p):
        print(f"💾 [磁盘缓存命中] 加载 preds_I{horizon} ← {p}")
        return np.load(p)
    return None


def _save_pred_cache(preds, horizon):
    p = _pred_cache_path(horizon)
    np.save(p, np.asarray(preds, dtype=np.float32))
    print(f"💾 [预测已缓存] preds_I{horizon} → {p}")

# ---------- I5 回测 ----------
if 'models_5' in locals() and len(models_5) >= 1:
    daily_ret_test_5 = _load_bt_cache(5)
    if 'preds_5' in locals():
        pass
    else:
        preds_5 = _load_pred_cache(5)
        if preds_5 is None:
            preds_5 = predict_ensemble(models_5, ds_test_5)
            _save_pred_cache(preds_5, 5)

    if daily_ret_test_5 is not None:
        df_test_valid_5 = df_test_5.reset_index(drop=True).iloc[ds_test_5.valid_indices].copy()
        df_test_valid_5['pred_score'] = preds_5
        print("⚡️ [缓存命中] 回测序列来自磁盘；已补齐 pred_score 供后续单元使用。")
    else:
        print("\n>>>>>>> [进行 5天期 I5 网络 评估] <<<<<<<")
        df_test_valid_5 = df_test_5.reset_index(drop=True).iloc[ds_test_5.valid_indices].copy()
        df_test_valid_5['pred_score'] = preds_5
        daily_ret_test_5 = evaluate_strategy(df_test_valid_5, F_horizon=5, df_daily=df_test_5)
        _save_bt_cache(daily_ret_test_5, 5)

# ---------- I20 回测 ----------
if 'models_20' in locals() and len(models_20) >= 1:
    daily_ret_test_20 = _load_bt_cache(20)
    if 'preds_20' in locals():
        pass
    else:
        preds_20 = _load_pred_cache(20)
        if preds_20 is None:
            preds_20 = predict_ensemble(models_20, ds_test_20)
            _save_pred_cache(preds_20, 20)

    if daily_ret_test_20 is not None:
        df_test_valid_20 = df_test_20.reset_index(drop=True).iloc[ds_test_20.valid_indices].copy()
        df_test_valid_20['pred_score'] = preds_20
        print("⚡️ [缓存命中] 回测序列来自磁盘；已补齐 pred_score 供后续单元使用。")
    else:
        print("\n>>>>>>> [进行 20天期 I20 网络 评估] <<<<<<<")
        df_test_valid_20 = df_test_20.reset_index(drop=True).iloc[ds_test_20.valid_indices].copy()
        df_test_valid_20['pred_score'] = preds_20
        daily_ret_test_20 = evaluate_strategy(df_test_valid_20, F_horizon=20, df_daily=df_test_20)
        _save_bt_cache(daily_ret_test_20, 20)

# ---------- I60 回测 ----------
if 'models_60' in locals() and len(models_60) >= 1:
    daily_ret_test_60 = _load_bt_cache(60)
    if 'preds_60' in locals():
        pass
    else:
        preds_60 = _load_pred_cache(60)
        if preds_60 is None:
            preds_60 = predict_ensemble(models_60, ds_test_60)
            _save_pred_cache(preds_60, 60)

    if daily_ret_test_60 is not None:
        df_test_valid_60 = df_test_60.reset_index(drop=True).iloc[ds_test_60.valid_indices].copy()
        df_test_valid_60['pred_score'] = preds_60
        print("⚡️ [缓存命中] 回测序列来自磁盘；已补齐 pred_score 供后续单元使用。")
    else:
        print("\n>>>>>>> [进行 60天期 I60 网络 评估] <<<<<<<")
        df_test_valid_60 = df_test_60.reset_index(drop=True).iloc[ds_test_60.valid_indices].copy()
        df_test_valid_60['pred_score'] = preds_60
        daily_ret_test_60 = evaluate_strategy(df_test_valid_60, F_horizon=60, df_daily=df_test_60)
        _save_bt_cache(daily_ret_test_60, 60)


## 六、导出 PPT 用素材

将 Accuracy/AUC、简易图等导出到 `OUTPUT_DIR/ppt_images`（若上游已产生 `preds_*` 等变量）。

In [ ]:
# ==========================================
# 步骤 11：一键生成 PPT 核心素材 (Results & Analysis)
# ==========================================
import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.metrics import roc_auc_score, accuracy_score

# 获取当前工作目录
PPT_OUT = os.path.join(OUTPUT_DIR, 'ppt_images')
os.makedirs(PPT_OUT, exist_ok=True)

print(f"====== 开始导出 PPT 素材至: {PPT_OUT} ======")

# --- PPT 16: 预测精度验证 (Accuracy & AUC) ---
metrics_list = []


def get_metrics_val(preds, ds_test, name):
    y_true = np.array([ds_test.labels[idx] for idx in ds_test.valid_indices], dtype=int)
    preds = np.asarray(preds, dtype=float).reshape(-1)
    n = len(y_true)
    if n == 0 or preds.size == 0:
        print(f"[{name}] 跳过指标：无有效样本（valid={n}, preds={preds.size}），常见于 smoke 窗口过短")
        return {"Model": name, "Accuracy": "N/A", "AUC-ROC": "N/A"}
    if preds.size != n:
        print(f"[{name}] 跳过指标：preds 长度 {preds.size} 与 y_true {n} 不一致")
        return {"Model": name, "Accuracy": "N/A", "AUC-ROC": "N/A"}
    acc = accuracy_score(y_true, (preds > 0.5).astype(int))
    auc_str = "N/A"
    if len(np.unique(y_true)) >= 2:
        try:
            auc_str = f"{roc_auc_score(y_true, preds):.4f}"
        except ValueError as err:
            print(f"[{name}] AUC 无法计算: {err}")
    else:
        print(f"[{name}] 标签仅单一类别，AUC 跳过")
    return {"Model": name, "Accuracy": f"{acc:.4%}", "AUC-ROC": auc_str}


for _pname, _dsname, _label in [
    ("preds_5", "ds_test_5", "I5 CNN"),
    ("preds_20", "ds_test_20", "I20 CNN"),
    ("preds_60", "ds_test_60", "I60 CNN"),
]:
    if _pname not in globals() or _dsname not in globals():
        continue
    metrics_list.append(get_metrics_val(globals()[_pname], globals()[_dsname], _label))

if metrics_list:
    df_metrics = pd.DataFrame(metrics_list)
    df_metrics.to_csv(os.path.join(PPT_OUT, 'ppt16_classification_metrics.csv'), index=False)
    print("- 已保存 PPT 16 数据表: ppt16_classification_metrics.csv")

# --- PPT 15: 累计收益曲线（分别导出 L5/L20/L60） ---
for _h in [5, 20, 60]:
    _k = f'daily_ret_test_{_h}'
    if _k not in globals():
        continue

    target_ret = globals()[_k]
    title_suffix = f"L={_h}"

    # 净收益（与 evaluate_strategy 一致：已扣 fee_rate×2）
    plt.figure(figsize=(10, 5))
    plt.plot(target_ret.index, (1 + target_ret['CNN_LS_net_EW']).cumprod(), label='CNN Portfolio EW (Net)', color='#c0392b', linewidth=2)
    plt.plot(target_ret.index, (1 + target_ret['Mom_LS_net']).cumprod(), label='Momentum Baseline (Net)', color='#7f8c8d', linestyle='--')
    plt.title(f'Strategy Cumulative Returns — Net of Costs ({title_suffix})', fontsize=12)
    plt.yscale('log')
    plt.legend()
    plt.grid(alpha=0.3)
    plt.tight_layout()
    net_name = f'ppt15_cumulative_returns_L{_h}.png'
    plt.savefig(os.path.join(PPT_OUT, net_name), dpi=300)
    plt.show()
    print(f"- 已保存 PPT 15（净收益/含成本）: {net_name}")

    # 毛收益（不计交易成本）
    plt.figure(figsize=(10, 5))
    plt.plot(target_ret.index, (1 + target_ret['CNN_LS_gross_EW']).cumprod(), label='CNN Portfolio EW (Gross)', color='#c0392b', linewidth=2)
    plt.plot(target_ret.index, (1 + target_ret['Mom_LS_gross']).cumprod(), label='Momentum Baseline (Gross)', color='#7f8c8d', linestyle='--')
    plt.title(f'Strategy Cumulative Returns — Gross of Costs ({title_suffix})', fontsize=12)
    plt.yscale('log')
    plt.legend()
    plt.grid(alpha=0.3)
    plt.tight_layout()
    gross_name = f'ppt15_cumulative_returns_gross_L{_h}.png'
    plt.savefig(os.path.join(PPT_OUT, gross_name), dpi=300)
    plt.show()
    print(f"- 已保存 PPT 15（毛收益/不含成本）: {gross_name}")


# --- PPT 19: 显著性热力图 ---
if 'plot_saliency_map' in dir() and 'models_60' in globals() and 'ds_test_60' in globals():
    print("- 正在生成 PPT 19 热力图素材...")
    plot_saliency_map(globals()['models_60'][0], globals()['ds_test_60'], num_samples=1)
    plt.gcf().savefig(os.path.join(PPT_OUT, 'ppt19_saliency_map.png'), dpi=300)
    print("- 已保存 PPT 19 图片: ppt19_saliency_map.png")
else:
    print("[跳过] PPT 19 热力图：plot_saliency_map 函数尚未定义，将在后续 cell 执行")

print("====== 导出过程结束 ======")


## 七、Fama–French 六因子剥离回归（可选）

下载 FF 因子并对多空组合收益做回归（需联网与 `pandas_datareader`）。

In [ ]:
import pandas as pd
# ==========================================
# 步骤 9：因子剥离回归分析 (Factor Spanning Tests / Alpha)
# ==========================================
import pandas_datareader.data as web
import statsmodels.api as sm
import warnings

warnings.filterwarnings('ignore', category=FutureWarning)


def _strip_columns(df):
    df = df.copy()
    df.columns = [str(c).strip() for c in df.columns]
    return df


def run_factor_regression(period_ret, F_horizon=5):
    """
    使用 Fama-French 5 因子 + Momentum 因子对我们的多空组合净收益 (CNN_LS_net & Mom_LS_net) 进行回归，
    验证其 Alpha 是否在剥离了市场和传统因子后依然显著。
    """
    print(f"\n====== Fama-French 6-Factor 剥离回归 (L={F_horizon}) ======")
    if period_ret is None or len(period_ret) == 0:
        print("period_ret 为空，跳过因子回归。")
        return

    pr = period_ret.copy()
    pr.index = pd.to_datetime(pr.index, utc=False).tz_localize(None).normalize()

    start_date = pr.index.min()
    end_date = pr.index.max()
    # 下载区间略放宽，避免边界日无行情
    pad = pd.Timedelta(days=35)
    dl_start = start_date - pad
    dl_end = end_date + pad

    try:
        ff5 = web.DataReader(
            'F-F_Research_Data_5_Factors_2x3_daily', 'famafrench', start=dl_start, end=dl_end
        )[0]
        mom = web.DataReader(
            'F-F_Momentum_Factor_daily', 'famafrench', start=dl_start, end=dl_end
        )[0]
    except Exception as e:
        print("网络下载 Fama-French 因子失败，请检查网络或配置代理。", e)
        return

    ff5 = _strip_columns(ff5)
    mom = _strip_columns(mom)
    # 动量因子列名在不同版本里可能是 Mom / WML 等
    if 'Mom' not in mom.columns and len(mom.columns) == 1:
        mom.columns = ['Mom']
    elif 'WML' in mom.columns and 'Mom' not in mom.columns:
        mom = mom.rename(columns={'WML': 'Mom'})

    ff6_daily = ff5.join(mom, how='inner') / 100.0
    ff6_daily.index = pd.to_datetime(ff6_daily.index, utc=False).tz_localize(None).normalize()
    ff6_daily = ff6_daily.sort_index()

    dates = pr.index
    ff6_period = []

    for i in range(len(dates)):
        t_start = pd.Timestamp(dates[i])
        t_end = pd.Timestamp(dates[i + 1]) if i + 1 < len(dates) else t_start + pd.Timedelta(days=max(14, int(F_horizon * 3)))
        cut = ff6_daily[(ff6_daily.index >= t_start) & (ff6_daily.index < t_end)]
        if len(cut) > 0:
            compounded = (1 + cut).prod() - 1
            compounded.name = t_start
            ff6_period.append(compounded)

    used_fallback = False
    if len(ff6_period) == 0:
        print(
            "[FF回归] 按调仓区间复利因子为空（常见于索引时区/日历未对齐）。"
            "退回：在每个调仓日用当日 FF 日因子（近似，非区间内复利）。"
        )
        used_fallback = True
        aligned = ff6_daily.reindex(dates, method='ffill')
        ff6_df = aligned.copy()
        ff6_df.index = dates
    else:
        # DataFrame(list_of_Series) 会得到 RangeIndex，丢失调仓日，无法与 pr 的日期索引 join
        ff6_df = pd.DataFrame({s.name: s for s in ff6_period}).T
        ff6_df.index = pd.to_datetime(ff6_df.index, utc=False).tz_localize(None).normalize()

    ff6_df = _strip_columns(ff6_df)

    reg_data = pr[['CNN_LS_net', 'Mom_LS_net']].join(ff6_df, how='inner').dropna(how='any')

    factor_names = ['Mkt-RF', 'SMB', 'HML', 'RMW', 'CMA', 'Mom']
    if len(reg_data) < max(15, len(factor_names) + 5):
        print(f"[FF回归] 对齐后样本过少 (n={len(reg_data)})，跳过回归。")
        return
    missing_yrf = [c for c in factor_names + ['RF'] if c not in reg_data.columns]
    if missing_yrf:
        print("[FF回归] 合并后缺少列:", missing_yrf)
        print("    当前列:", list(reg_data.columns))
        print("    ff6_daily 列:", list(ff6_daily.columns))
        return

    X = reg_data[factor_names]
    X = sm.add_constant(X)

    # 持有期对应的无风险复利（若未做 fallback 则已按区间复利；fallback 下 RF 为调仓日对齐的日度 RF）
    rf_series = reg_data['RF']

    y_cnn = reg_data['CNN_LS_net'] - rf_series
    y_mom = reg_data['Mom_LS_net'] - rf_series

    model_cnn = sm.OLS(y_cnn, X).fit()
    model_mom = sm.OLS(y_mom, X).fit()

    from IPython.display import display

    print("\n>>>  CNN 策略 FF6 因子回归结果  <<<")
    if used_fallback:
        print("（提示：使用了调仓日对齐近似因子，系数仅供管线自检，勿与论文严格对比。）")
    ppy = PERIODS_PER_YEAR[F_horizon]
    ann_alpha_cnn = model_cnn.params['const'] * ppy
    print(f"[CNN 年化超额 Alpha]: {ann_alpha_cnn * 100:.2f}% (t-value: {model_cnn.tvalues['const']:.2f})")
    display(model_cnn.summary().tables[1])

    print("\n>>> Baseline 动量策略 FF6 因子回归结果  <<<")
    ann_alpha_mom = model_mom.params['const'] * ppy
    print(f"[Mom 年化超额 Alpha]: {ann_alpha_mom * 100:.2f}% (t-value: {model_mom.tvalues['const']:.2f})")
    display(model_mom.summary().tables[1])


# 执行分析 (如果回测变量已生成)
if 'daily_ret_test_60' in locals():
    run_factor_regression(daily_ret_test_60, F_horizon=60)
elif 'daily_ret_test_20' in locals():
    run_factor_regression(daily_ret_test_20, F_horizon=20)
elif 'daily_ret_test_5' in locals():
    run_factor_regression(daily_ret_test_5, F_horizon=5)


## 八、模型可解释性：Saliency Map（可选）

对高置信正例绘制输入梯度热力图，观察 CNN 关注的 K 线区域。

In [ ]:
# ==========================================
# 步骤 10：模型可解释性 - 显著性热力图 (Saliency Maps)
# 对应论文中的“为什么CNN看到了趋势” —— 打开深度学习的黑盒
# ==========================================

def plot_saliency_map(model, dataset, num_samples=3):
    """
    通过计算输出得分对输入像素的梯度 (Gradients)，
    来可视化 CNN 模型究竟“盯着”图表上的哪些像素做出了预测。
    """
    import matplotlib.colors as mcolors
    print("\n====== 生成 CNN 视觉显著性热力图 (Saliency Map) ======")
    
    model.eval()
    
    # 找几个模型预测非常有信心的样本 (只选预测上涨且标签也是上涨的 True Positive 绝佳形态)
    found = 0
    indices_to_plot = []
    
    # 随机打乱搜索，避免每次都是前面的无聊横盘图
    # 这里移除固定的 seed 以便每次运行都能刷出不同的股票（从而找到K线更完美的票）
    search_idxs = np.arange(len(dataset))
    np.random.shuffle(search_idxs)
    
    for idx in search_idxs:
        X_tensor, y_label = dataset[idx]
        if y_label.item() != 1:
            continue
            
        X_unsqueeze = X_tensor.unsqueeze(0).to(device)
        with torch.no_grad():
            outputs = model(X_unsqueeze)
            prob = torch.nn.functional.softmax(outputs, dim=1)[0, 1].item()
            
        # 寻找预测上涨概率极高 (自信) 的形态
        if prob > 0.85:
            # 额外加一个极其暴力的视觉过滤：如果这张图因为流动性太差导致像素全是横盘一条直线/大面积留白（标准差太小），则无情跳过！
            img_std = X_tensor.std().item()
            if img_std > 0.25: # 只抓取那些形态非常丰富、上下翻飞且带有巨量成交柱的股票 K 线！
                indices_to_plot.append((idx, prob))
                found += 1
            
            if found >= num_samples:
                break
                
    if not indices_to_plot:
        print("未能在前几百个样本中找到极其置信的形态，直接使用随机正例样本。")
        indices_to_plot = [(i, 0.0) for i in range(num_samples)]
        
    fig, axes = plt.subplots(num_samples, 2, figsize=(14, 4 * num_samples))
    if num_samples == 1: axes = [axes]
    
    for i, (idx, prob) in enumerate(indices_to_plot):
        X_tensor, y_label = dataset[idx]
        
        # 开启梯度追踪
        X_input = X_tensor.unsqueeze(0).to(device)
        X_input.requires_grad_()
        
        # 前向传播并针对“类别 1(上涨)”的分数求导
        outputs = model(X_input)
        score = outputs[0, 1] 
        score.backward()
        
        # 获取输入图像上的梯度绝对值
        saliency = X_input.grad.data.abs().squeeze().cpu().numpy()
        
        # 提取输入图片
        original_img = X_tensor.squeeze().cpu().numpy()
        
        # 将其还原为 0~1 的像素度阶 (如果 dataset 中存在均值/方差)
        if hasattr(dataset, 'mu') and hasattr(dataset, 'std') and dataset.mu is not None:
             original_img = original_img * dataset.std + dataset.mu
                
        # 颠倒颜色以让 K 线图背景变为白色，K线变为黑色，更易于审阅
        vibrant_img = 1.0 - original_img
        
        ax_orig = axes[i][0]
        ax_heat = axes[i][1]
        
        # 白底黑线的清晰版
        ax_orig.imshow(vibrant_img, cmap='gist_yarg', origin='upper')
        ax_orig.set_title(f"CNN Focus Extracted Image\nConfidence: {prob:.1%}", fontsize=12)
        ax_orig.axis('off')
        
        # 归一化热力图
        saliency_norm = (saliency - saliency.min()) / (saliency.max() - saliency.min() + 1e-8)
        
        # 叠加红色热力图
        ax_heat.imshow(vibrant_img, cmap='gist_yarg', origin='upper')
        im = ax_heat.imshow(saliency_norm, cmap='Reds', origin='upper', alpha=0.55)
        ax_heat.set_title(f"Saliency Map (Red spots)\nActual Label: {y_label.item()}", color='darkred', fontsize=12)
        ax_heat.axis('off')
        
        fig.colorbar(im, ax=ax_heat, fraction=0.046, pad=0.04)
        
    plt.tight_layout()
    plt.show()

# 提取 L=60 天 (最有形态和趋势感的长窗口) 模型来看最具有冲击力
if 'models_60' in locals() and len(models_60) > 0 and 'ds_test_60' in locals():
    plot_saliency_map(models_60[0], ds_test_60, num_samples=3)
else:
    print("模型60天未加载，请先执行前置训练与测试代码。")


## 九、论文风格图表（Figure 6 / 7 / 8、Table I）

以下单元依赖前文中的 `PERIODS_PER_YEAR`、`enrich_rebalance_panel_with_daily_baselines`、`df_test_*`、`df_daily` 等。

### 9.0 可选准备：加载已保存模型与累计收益工具

若需单独调试图表，可从磁盘加载 I20 权重；下一单元提供累计收益曲线绘制函数。

In [ ]:
import os
import torch
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path
from torch.utils.data import DataLoader

# ==========================================
# 1. 设备与路径（与训练单元一致：MODEL_DIR / models_smoke）
# ==========================================
model_dir = globals().get("MODEL_DIR", os.path.join(globals().get("OUTPUT_DIR", str(Path.cwd() / "outputs")), "models"))
if "select_device" in globals():
    device = select_device()
elif torch.cuda.is_available():
    device = torch.device("cuda")
elif hasattr(torch.backends, "mps") and torch.backends.mps.is_available():
    device = torch.device("mps")
else:
    device = torch.device("cpu")
PIN_MEMORY = globals().get("PIN_MEMORY", device.type == "cuda")
print(f"✅ 正在使用设备: {device}")


def _load_checkpoint_state_dict(path, map_location):
    ckpt = torch.load(path, map_location=map_location)
    if isinstance(ckpt, dict) and "model_state_dict" in ckpt:
        return ckpt["model_state_dict"]
    return ckpt


def _find_i20_checkpoint_dir(d):
    seeds = [42]
    if "_training_seeds" in globals():
        seeds = list(_training_seeds())
    for seed in seeds:
        p = os.path.join(d, f"I20_checkpoint{seed}.pth.tar")
        if os.path.isfile(p):
            return p
    return None


# ==========================================
# 2. 模型：优先用当前会话已训练的 models_20；否则从 MODEL_DIR 加载 .pth.tar
#    （不再使用 I20_seed42.pt，且 load_state_dict 必须用 model_state_dict）
# ==========================================
model_instance = None
loaded_from = None

if "models_20" in globals() and isinstance(globals()["models_20"], list) and len(globals()["models_20"]) > 0:
    model_instance = globals()["models_20"][0]
    model_instance = model_instance.to(device).eval()
    loaded_from = "内存中的 models_20[0]（跳过磁盘覆盖）"
    print(f"✅ {loaded_from}")
elif "I20Model" in globals():
    model_instance = globals()["I20Model"]().to(device)
    ckpt_path = _find_i20_checkpoint_dir(model_dir)
    if ckpt_path is None:
        print(
            f"❌ 未在 {model_dir} 找到 I20_checkpoint*.pth.tar。"
            "请先训练 I20（SMOKE 下需在 SMOKE_TASKS 中加入 (20,20)），或 SMOKE_TEST=False 全量训练。"
        )
        model_instance = None
    else:
        try:
            sd = _load_checkpoint_state_dict(ckpt_path, device)
            model_instance.load_state_dict(sd, strict=True)
            model_instance.eval()
            loaded_from = ckpt_path
            print(f"✅ 成功加载权重: {ckpt_path}")
        except Exception as e:
            print(
                f"⚠️ 权重与当前 I20Model 结构不一致（常见于旧 checkpoint 或不同代码版本）: {e}\n"
                "   请删除不匹配的旧文件或重新训练 I20，勿继续使用随机初始化做显著性分析。"
            )
            model_instance = None
else:
    print("❌ 错误：未找到 I20Model 定义")

# ==========================================
# 3. DataLoader：ds_test_20 是 Dataset，必须包一层 DataLoader
# ==========================================
all_preds = []
all_actuals = []
all_images = []
all_labels_for_img = []

target_loader = None
loader_name = None
if "test_loader_20" in globals() and isinstance(globals()["test_loader_20"], DataLoader):
    target_loader = globals()["test_loader_20"]
    loader_name = "test_loader_20"
elif "ds_test_20" in globals() and globals()["ds_test_20"] is not None:
    bs = int(globals().get("SMOKE_BATCH_SIZE", 128))
    target_loader = make_dataloader(
        globals()["ds_test_20"], batch_size=bs, shuffle=False
    )
    loader_name = f"DataLoader(ds_test_20, batch_size={bs})"
else:
    print(
        "❌ 未找到 ds_test_20 / test_loader_20。"
        "若仅跑了 SMOKE_TASKS=[(5,5)]，不会生成 I20 测试集；请加入 (20,20) 或关闭 SMOKE 后训练 I20。"
    )

if target_loader is not None and model_instance is not None:
    print(f"🚀 正在通过 {loader_name} 生成预测结果...")
    actual_device = next(model_instance.parameters()).device
    
    with torch.no_grad():
        for i, (images, labels) in enumerate(target_loader):
            images = images.to(actual_device, non_blocking=PIN_MEMORY)
            
            # 【维度修复 1】输入图像维度对齐
            if images.dim() == 3:
                images = images.unsqueeze(0)
            
            outputs = model_instance(images)
            probs = torch.nn.functional.softmax(outputs, dim=1)[:, 1]
            all_preds.extend(probs.detach().cpu().numpy().reshape(-1).tolist())
            
            # 【维度修复 2】处理 0-dim tensor 的 labels
            if labels.dim() == 0:
                all_actuals.append(labels.item())
            else:
                all_actuals.extend(labels.cpu().numpy().flatten())
            
            # 【核心修复 3】防御第 92 行的切片报错
            if i == 0:
                all_images = images[:5].detach()
                # 如果 labels 是标量，就把它包装成列表；如果是 Tensor 则正常切片
                if labels.dim() == 0:
                    all_labels_for_img = torch.tensor([labels.item()])
                else:
                    all_labels_for_img = labels[:5].detach()
            
            if i > 30: break 
            
    print(f"✅ 处理完成！已获取 {len(all_preds)} 个样本。")
else:
    if model_instance is None:
        print("❌ 跳过显著性/可视化流程：无有效 I20 权重（见上方日志）。")
    elif target_loader is None:
        print("❌ 跳过：无 I20 测试集 DataLoader（见上方日志）。")
    else:
        print("❌ 跳过：模型与数据未同时就绪。")

### 9.0（续）累计收益曲线

定义 `plot_strategy_cumulative_returns`：按预测分位分组后的多头/空头/多空累计收益可视化。

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import torch

# ==========================================
# 图表 1: 累计收益曲线 (自动处理长度不匹配)
# ==========================================
def plot_strategy_cumulative_returns(preds, actuals):
    if len(preds) == 0:
        print("⚠️ 没有预测数据")
        return
    
    # 对齐长度
    min_size = min(len(preds), len(actuals))
    preds_arr = np.array(preds[:min_size])
    actuals_arr = np.array(actuals[:min_size])
    
    # 划分多空
    p_high = np.percentile(preds_arr, 80)
    p_low = np.percentile(preds_arr, 20)
    
    long_mask = preds_arr >= p_high
    short_mask = preds_arr <= p_low
    
    plt.figure(figsize=(10, 5), dpi=120)
    
    # 多头与空头曲线
    if np.any(long_mask):
        plt.plot(np.cumsum(actuals_arr[long_mask]), label='Long Portfolio (Top 20%)', color='#2ca02c', alpha=0.6)
    if np.any(short_mask):
        plt.plot(np.cumsum(-actuals_arr[short_mask]), label='Short Portfolio (Bottom 20%)', color='#d62728', alpha=0.6)
    
    # 多空对冲 Alpha 曲线
    min_ls = min(sum(long_mask), sum(short_mask))
    if min_ls > 0:
        ls_ret = actuals_arr[long_mask][:min_ls] - actuals_arr[short_mask][:min_ls]
        plt.plot(np.cumsum(ls_ret), label='Long-Short Alpha', color='#1f77b4', linewidth=3)
    
    plt.title('Strategy Cumulative Returns (Reproduction)', fontweight='bold')
    plt.xlabel('Number of Trades')
    plt.ylabel('Cumulative Return')
    plt.legend()
    plt.grid(True, alpha=0.3)
    plt.show()

# ==========================================
# 图表 2: 显著性热力图 (修复索引报错)
# ==========================================
def plot_saliency_heatmaps(model, images, labels):
    if model is None or len(images) == 0:
        print("⚠️ 数据未准备好")
        return
        
    model.eval()
    actual_device = next(model.parameters()).device
    input_imgs = images.clone().detach().to(actual_device).requires_grad_(True)
    
    outputs = model(input_imgs)
    score = outputs.sum()
    model.zero_grad()
    score.backward()
    
    saliency, _ = torch.max(input_imgs.grad.data.abs(), dim=1)
    
    num_plots = min(3, len(images))
    fig, axes = plt.subplots(num_plots, 2, figsize=(10, num_plots * 3.5), dpi=100)
    
    # 【核心修复】处理 axes 可能是一维的情况
    if num_plots == 1:
        axes = np.expand_dims(axes, axis=0)
    
    for i in range(num_plots):
        img_np = input_imgs[i][0].detach().cpu().numpy()
        sal_np = saliency[i].cpu().numpy()
        sal_norm = (sal_np - sal_np.min()) / (sal_np.max() - sal_np.min() + 1e-8)
        
        # 左图：原始图
        axes[i, 0].imshow(img_np, cmap='gray')
        axes[i, 0].set_title(f"Sample {i+1} Original")
        axes[i, 0].axis('off')
        
        # 右图：热力图叠加
        axes[i, 1].imshow(img_np, cmap='gray', alpha=0.6)
        axes[i, 1].imshow(norm_sal := sal_norm, cmap='jet', alpha=0.5)
        axes[i, 1].set_title("CNN Saliency Focus")
        axes[i, 1].axis('off')
    
    plt.tight_layout()
    plt.show()

# 执行绘图
if 'all_preds' in locals() and len(all_preds) > 0:
    plot_strategy_cumulative_returns(all_preds, all_actuals)
    plot_saliency_heatmaps(model_instance, all_images, all_labels_for_img if 'all_labels_for_img' in locals() else None)

### 9.1 Figure 6 — 分位数预测精度（CNN vs 动量 vs 均线）

`plot_paper_figure6`：十分位年化收益与波动；需要 `df_daily` 合并传统基准。

In [ ]:
# ==========================================
# 论文 Figure 6 复现：分位数预测精度图
# 【修复版】内部补全基准分数计算，零依赖、零重跑
# ==========================================
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

FIG_OUT_DIR = os.environ.get("FIG_OUT_DIR", "/root/autodl-tmp/outputs/pipeline_runs/figures")
FIG_DATA_OUT_DIR = os.environ.get("FIG_DATA_OUT_DIR", "/root/autodl-tmp/outputs/pipeline_runs/tables")
os.makedirs(FIG_OUT_DIR, exist_ok=True)
os.makedirs(FIG_DATA_OUT_DIR, exist_ok=True)

def calc_decile_metrics(df_test_valid, F_horizon, score_col='pred_score'):
    """
    计算10分位数的平均年化收益、年化波动率
    完全匹配论文的计算逻辑：每日截面分组，时序平均
    """
    df = df_test_valid.copy()
    # 1. 每日截面按预测分，分成10组
    df['decile'] = df.groupby('dlycaldt')[score_col].rank(pct=True, ascending=True)
    df['decile'] = (df['decile'] * 10).apply(np.ceil).astype(int) # 1-10组
    
    # 2. 计算每组的平均收益、波动率
    decile_stats = df.groupby('decile')[f'R_fut_{F_horizon}'].agg(
        mean_ret='mean',
        std_ret='std'
    ).reset_index()
    
    # 3. 年化处理（与作者一致：每年 52/12/4 个持有期）
    ppy = PERIODS_PER_YEAR[F_horizon]
    decile_stats['ann_ret'] = decile_stats['mean_ret'] * ppy
    decile_stats['ann_vol'] = decile_stats['std_ret'] * np.sqrt(ppy)
    
    return decile_stats

def plot_paper_figure6(df_test_valid, F_horizon, title_suffix="", df_daily=None, save_path=None, csv_prefix=None):
    """
    完全复刻论文 Figure 6 的双图布局。
    周期末稀疏面板需传入 df_daily，在日频上计算动量/均线再 merge。
    """
    df = df_test_valid.copy()
    if df_daily is not None:
        df = enrich_rebalance_panel_with_daily_baselines(df, df_daily, F_horizon)
    else:
        df['mom_score'] = df.groupby('permno')['dlyclose_adj'].pct_change(F_horizon).fillna(0)
        ma_fast = df.groupby('permno')['dlyclose_adj'].transform(
            lambda x: x.rolling(max(2, F_horizon // 2), min_periods=1).mean()
        )
        ma_slow = df.groupby('permno')['dlyclose_adj'].transform(
            lambda x: x.rolling(max(20, F_horizon), min_periods=1).mean()
        )
        df['ma_score'] = (ma_fast / ma_slow - 1).fillna(0)
    
    # 1. 计算CNN模型和两个基准的分位数结果
    cnn_decile = calc_decile_metrics(df, F_horizon, score_col='pred_score')
    mom_decile = calc_decile_metrics(df, F_horizon, score_col='mom_score')
    ma_decile = calc_decile_metrics(df, F_horizon, score_col='ma_score')

    if csv_prefix is not None:
        cnn_decile.to_csv(os.path.join(FIG_DATA_OUT_DIR, f"{csv_prefix}_cnn.csv"), index=False)
        mom_decile.to_csv(os.path.join(FIG_DATA_OUT_DIR, f"{csv_prefix}_mom.csv"), index=False)
        ma_decile.to_csv(os.path.join(FIG_DATA_OUT_DIR, f"{csv_prefix}_ma.csv"), index=False)
        print(f"[Figure6] 已保存明细CSV前缀: {csv_prefix}_*.csv")

    # 2. 画图：和论文完全一致的双面板布局
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 7), dpi=120)
    decile_x = np.arange(1, 11) # 1-10分位数X轴
    
    # ---------------- 左图：分位数年化收益 ----------------
    ax1.plot(decile_x, cnn_decile['ann_ret'], 'o-', color='#d62728', linewidth=2, markersize=6, label='CNN (I{}/R{})'.format(F_horizon, F_horizon))
    ax1.plot(decile_x, mom_decile['ann_ret'], '--', color='#1f77b4', linewidth=1.5, markersize=5, label='Baseline Momentum')
    ax1.plot(decile_x, ma_decile['ann_ret'], '-.', color='#2ca02c', linewidth=1.5, markersize=5, label='Baseline MA Cross')
    
    ax1.set_title('Average Realized Return by Decile', fontsize=12, fontweight='bold')
    ax1.set_xlabel('Signal Decile', fontsize=11)
    ax1.set_ylabel('Annualized Return', fontsize=11)
    ax1.set_xticks(decile_x)
    ax1.grid(True, alpha=0.3, linestyle='--')
    ax1.legend(fontsize=10)
    ax1.axhline(y=0, color='black', linestyle='-', alpha=0.5) # 0收益基准线
    
    # ---------------- 右图：分位数年化波动率 ----------------
    ax2.plot(decile_x, cnn_decile['ann_vol'], 'o-', color='#d62728', linewidth=2, markersize=6, label='CNN (I{}/R{})'.format(F_horizon, F_horizon))
    ax2.plot(decile_x, mom_decile['ann_vol'], '--', color='#1f77b4', linewidth=1.5, markersize=5, label='Baseline Momentum')
    ax2.plot(decile_x, ma_decile['ann_vol'], '-.', color='#2ca02c', linewidth=1.5, markersize=5, label='Baseline MA Cross')
    
    ax2.set_title('Return Volatility by Decile', fontsize=12, fontweight='bold')
    ax2.set_xlabel('Signal Decile', fontsize=11)
    ax2.set_ylabel('Annualized Volatility', fontsize=11)
    ax2.set_xticks(decile_x)
    ax2.grid(True, alpha=0.3, linestyle='--')
    ax2.legend(fontsize=10)
    
    # 总标题，和论文对齐
    fig.suptitle('Figure 6. Prediction Accuracy by Decile {}'.format(title_suffix), fontsize=14, fontweight='bold')
    plt.tight_layout()
    if save_path is not None:
        plt.savefig(save_path, dpi=300)
        print(f"[Figure6] 已保存: {save_path}")
    plt.show()
    
    # 打印数值结果，汇报用
    print(f"\n====== I{F_horizon}/R{F_horizon} 分位数年化收益 ======")
    print(cnn_decile[['decile', 'ann_ret', 'ann_vol']].to_string(index=False))

# ==========================================
# 一键生成3个周期的图（和你之前的代码完全适配）
# ==========================================
FIG6_ENABLE = os.environ.get("FIG6_ENABLE", "0").strip().lower() in {"1", "true", "yes", "on"}
# 生成5天周期的图（和论文的I5/R5完全对应）
if FIG6_ENABLE and 'df_test_valid_5' in locals():
    plot_paper_figure6(
        df_test_valid_5,
        F_horizon=5,
        title_suffix="(I5/R5)",
        df_daily=df_test_5 if 'df_test_5' in locals() else None,
        save_path=os.path.join(FIG_OUT_DIR, "fig6_I5.png"),
        csv_prefix="fig6_I5",
        )

# 生成20天周期的图（你的王牌模型）
if FIG6_ENABLE and 'df_test_valid_20' in locals():
    plot_paper_figure6(
            df_test_valid_20,
            F_horizon=20,
            title_suffix="(I20/R20)",
            df_daily=df_test_20 if 'df_test_20' in locals() else None,
            save_path=os.path.join(FIG_OUT_DIR, "fig6_I20.png"),
            csv_prefix="fig6_I20",
        )

# 生成60天周期的图
if FIG6_ENABLE and 'df_test_valid_60' in locals():
    plot_paper_figure6(
            df_test_valid_60,
            F_horizon=60,
            title_suffix="(I60/R60)",
            df_daily=df_test_60 if 'df_test_60' in locals() else None,
            save_path=os.path.join(FIG_OUT_DIR, "fig6_I60.png"),
            csv_prefix="fig6_I60",
        )

if not FIG6_ENABLE:
    print("[Figure6] 已按 FIG6_ENABLE=0 跳过")

### 9.2 Figure 7（简化）— CNN 与传统基准分位数对比

`plot_paper_figure7_simple`。

In [ ]:
# ==========================================
# 论文 Figure7 简化版复现：核心模型vs传统模型对比
# 零重跑、零训练，直接用你已有的数据生成
# ==========================================
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

FIG_OUT_DIR = os.environ.get("FIG_OUT_DIR", "/root/autodl-tmp/outputs/pipeline_runs/figures")
FIG_DATA_OUT_DIR = os.environ.get("FIG_DATA_OUT_DIR", "/root/autodl-tmp/outputs/pipeline_runs/tables")
os.makedirs(FIG_OUT_DIR, exist_ok=True)
os.makedirs(FIG_DATA_OUT_DIR, exist_ok=True)

def calc_decile_metrics_simple(df_test_valid, F_horizon, score_col):
    """简化版分位数计算，和论文逻辑完全一致"""
    df = df_test_valid.copy()
    # 每日截面按信号分成10组
    df['decile'] = df.groupby('dlycaldt')[score_col].rank(pct=True, ascending=True)
    df['decile'] = (df['decile'] * 10).apply(np.ceil).astype(int)
    # 计算年化收益、波动率
    ppy = PERIODS_PER_YEAR[F_horizon]
    decile_stats = df.groupby('decile')[f'R_fut_{F_horizon}'].agg(
        mean_ret='mean',
        std_ret='std'
    ).reset_index()
    decile_stats['ann_ret'] = decile_stats['mean_ret'] * ppy
    decile_stats['ann_vol'] = decile_stats['std_ret'] * np.sqrt(ppy)
    return decile_stats

def plot_paper_figure7_simple(df_test_valid, F_horizon, title_suffix="", df_daily=None, save_path=None, csv_prefix=None):
    """
    简化版Figure7，完全贴合论文对比逻辑
    对比：你的2D CNN核心模型 vs 3个论文核心传统基准模型
    """
    df = df_test_valid.copy()
    if df_daily is not None:
        df = enrich_rebalance_panel_with_daily_baselines(df, df_daily, F_horizon)
    else:
        df['mom_score'] = df.groupby('permno')['dlyclose_adj'].pct_change(F_horizon).fillna(0)
        ma_fast = df.groupby('permno')['dlyclose_adj'].transform(
            lambda x: x.rolling(max(2, F_horizon // 2), min_periods=1).mean()
        )
        ma_slow = df.groupby('permno')['dlyclose_adj'].transform(
            lambda x: x.rolling(max(20, F_horizon), min_periods=1).mean()
        )
        df['ma_score'] = (ma_fast / ma_slow - 1).fillna(0)
        df['reversal_score'] = -df.groupby('permno')['dlyclose_adj'].pct_change(5).fillna(0)
    
    # 计算所有模型的分位数结果
    cnn_decile = calc_decile_metrics_simple(df, F_horizon, 'pred_score')
    mom_decile = calc_decile_metrics_simple(df, F_horizon, 'mom_score')
    ma_decile = calc_decile_metrics_simple(df, F_horizon, 'ma_score')
    reversal_decile = calc_decile_metrics_simple(df, F_horizon, 'reversal_score')

    if csv_prefix is not None:
        cnn_decile.to_csv(os.path.join(FIG_DATA_OUT_DIR, f"{csv_prefix}_cnn.csv"), index=False)
        reversal_decile.to_csv(os.path.join(FIG_DATA_OUT_DIR, f"{csv_prefix}_reversal.csv"), index=False)
        mom_decile.to_csv(os.path.join(FIG_DATA_OUT_DIR, f"{csv_prefix}_mom.csv"), index=False)
        ma_decile.to_csv(os.path.join(FIG_DATA_OUT_DIR, f"{csv_prefix}_ma.csv"), index=False)
        print(f"[Figure7-simple] 已保存明细CSV前缀: {csv_prefix}_*.csv")

    # 画图：和论文完全一致的双面板布局
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 7), dpi=120)
    decile_x = np.arange(1, 11) # 1-10分位数X轴
    
    # ---------------- 左图：分位数年化收益（和论文左图对应） ----------------
    ax1.plot(decile_x, cnn_decile['ann_ret'], 'o-', color='#d62728', linewidth=2, markersize=6, label='CNN 2D (Image Scale)')
    ax1.plot(decile_x, reversal_decile['ann_ret'], 'o--', color='#9467bd', linewidth=1.5, markersize=5, label='Short-Term Reversal')
    ax1.plot(decile_x, mom_decile['ann_ret'], 'o--', color='#1f77b4', linewidth=1.5, markersize=5, label='Momentum')
    ax1.plot(decile_x, ma_decile['ann_ret'], 'o--', color='#2ca02c', linewidth=1.5, markersize=5, label='MA Cross')
    
    ax1.set_title('Average Realized Return by Decile', fontsize=12, fontweight='bold')
    ax1.set_xlabel('Signal Decile', fontsize=11)
    ax1.set_ylabel('Annualized Return', fontsize=11)
    ax1.set_xticks(decile_x)
    ax1.grid(True, alpha=0.3, linestyle='--')
    ax1.legend(fontsize=10)
    ax1.axhline(y=0, color='black', linestyle='-', alpha=0.5)
    
    # ---------------- 右图：分位数年化波动率（和论文右图对应） ----------------
    ax2.plot(decile_x, cnn_decile['ann_vol'], 'o-', color='#d62728', linewidth=2, markersize=6, label='CNN 2D (Image Scale)')
    ax2.plot(decile_x, reversal_decile['ann_vol'], 'o--', color='#9467bd', linewidth=1.5, markersize=5, label='Short-Term Reversal')
    ax2.plot(decile_x, mom_decile['ann_vol'], 'o--', color='#1f77b4', linewidth=1.5, markersize=5, label='Momentum')
    ax2.plot(decile_x, ma_decile['ann_vol'], 'o--', color='#2ca02c', linewidth=1.5, markersize=5, label='MA Cross')
    
    ax2.set_title('Return Volatility by Decile', fontsize=12, fontweight='bold')
    ax2.set_xlabel('Signal Decile', fontsize=11)
    ax2.set_ylabel('Annualized Volatility', fontsize=11)
    ax2.set_xticks(decile_x)
    ax2.grid(True, alpha=0.3, linestyle='--')
    ax2.legend(fontsize=10)
    
    # 总标题，和论文对齐
    fig.suptitle('Figure:  Model Prediction Accuracy by Decile {}'.format(title_suffix), fontsize=14, fontweight='bold')
    plt.tight_layout()
    if save_path is not None:
        plt.savefig(save_path, dpi=300)
        print(f"[Figure7-simple] 已保存: {save_path}")
    plt.show()

# ==========================================
# 一键生成你的王牌模型（20天周期）的图
# 优先用20天模型，你的效果最好，汇报最亮眼
# ==========================================
FIG7_ENABLE = os.environ.get("FIG7_ENABLE", "0").strip().lower() in {"1", "true", "yes", "on"}
if FIG7_ENABLE:
    for _h in (5, 20, 60):
        _valid_key = f"df_test_valid_{_h}"
        _daily_key = f"df_test_{_h}"
        if _valid_key in locals():
            plot_paper_figure7_simple(
                locals()[_valid_key],
                F_horizon=_h,
                title_suffix=f"(I{_h}/R{_h})",
                df_daily=locals()[_daily_key] if _daily_key in locals() else None,
                save_path=os.path.join(FIG_OUT_DIR, f"fig7_simple_I{_h}.png"),
                csv_prefix=f"fig7_simple_I{_h}",
            )
else:
    print("[Figure7-simple] 已按 FIG7_ENABLE=0 跳过")

### 9.3 Figure 7（完整）— CNN vs Logistic vs 传统基准

`plot_paper_figure7_fixed`。

In [ ]:
# ==========================================
# 论文 Figure7 修复版复现：CNN vs Linear Model vs 传统基准
# 【完整修复版】解决Linear Model波动率陡升问题，完全贴合论文
# 零重跑CNN、直接运行、10秒出图
# ==========================================
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler

FIG_OUT_DIR = os.environ.get("FIG_OUT_DIR", "/root/autodl-tmp/outputs/pipeline_runs/figures")
FIG_DATA_OUT_DIR = os.environ.get("FIG_DATA_OUT_DIR", "/root/autodl-tmp/outputs/pipeline_runs/tables")
os.makedirs(FIG_OUT_DIR, exist_ok=True)
os.makedirs(FIG_DATA_OUT_DIR, exist_ok=True)

def calc_decile_metrics_full(df_test_valid, F_horizon, score_col):
    """完整版分位数计算，和论文逻辑完全一致"""
    df = df_test_valid.copy()
    # 每日截面按信号分成10组
    df['decile'] = df.groupby('dlycaldt')[score_col].rank(pct=True, ascending=True)
    df['decile'] = (df['decile'] * 10).apply(np.ceil).astype(int)
    # 计算年化收益、波动率
    ppy = PERIODS_PER_YEAR[F_horizon]
    decile_stats = df.groupby('decile')[f'R_fut_{F_horizon}'].agg(
        mean_ret='mean',
        std_ret='std'
    ).reset_index()
    decile_stats['ann_ret'] = decile_stats['mean_ret'] * ppy
    decile_stats['ann_vol'] = decile_stats['std_ret'] * np.sqrt(ppy)
    return decile_stats

def plot_paper_figure7_fixed(df_test_valid, F_horizon, title_suffix="", df_daily=None, save_path=None, csv_prefix=None):
    """
    修复版Figure7，完全贴合论文
    【核心修复】优化Linear Model特征、增加极值处理，解决波动率陡升问题
    """
    df = df_test_valid.copy()
    if df_daily is None:
        raise ValueError("plot_paper_figure7_fixed 需要 df_daily，以便在日频上构造动量/均线/反转与 Linear 特征")

    df = enrich_rebalance_panel_with_daily_baselines(df, df_daily, F_horizon)
    df = merge_linear_features_from_daily(df, df_daily)

    fig7_cache = os.path.join(CACHE_DIR, f"figure7_deciles_I{F_horizon}.pkl")
    fig7_force = os.environ.get("FIG7_FORCE_REBUILD", "0").strip().lower() in {"1", "true", "yes", "on"}
    if os.path.exists(fig7_cache) and not fig7_force:
        print(f"💾 [磁盘缓存命中] Figure7 deciles ← {fig7_cache}")
        cached = pd.read_pickle(fig7_cache)
        cnn_decile = cached['cnn_decile']
        linear_decile = cached['linear_decile']
        mom_decile = cached['mom_decile']
        ma_decile = cached['ma_decile']
        reversal_decile = cached['reversal_decile']
    else:
        print("====== 训练对齐作者的 Linear Model（稳态内存版） ======")

        # 控制特征维度与样本量，避免 notebook 后段内核被 OOM 杀死
        max_lag = int(os.environ.get("FIG7_MAX_LAG", "6"))
        max_train_rows = int(os.environ.get("FIG7_MAX_TRAIN_ROWS", "400000"))
        pred_chunk = int(os.environ.get("FIG7_PRED_CHUNK", "200000"))

        L = MODEL_CFG[F_horizon]['W'] // 3
        lag_limit = max(1, min(L, max_lag))
        lag_cols = []

        # 在调仓截面内做 lag 特征，避免在全量日频面板上生成超大临时矩阵
        df = df.sort_values(['permno', 'dlycaldt']).copy()
        for lag in range(1, lag_limit + 1):
            for col_base, src_col in [('o', 'dlyopen_adj'), ('h', 'dlyhigh_adj'),
                                       ('l', 'dlylow_adj'), ('c', 'dlyclose_adj'),
                                       ('v', 'dlyvol')]:
                feat_name = f'{col_base}_lag{lag}'
                df[feat_name] = df.groupby('permno')[src_col].shift(lag)
                lag_cols.append(feat_name)

            ma_col = f'ma_lag{lag}'
            df[ma_col] = (
                df.groupby('permno')['dlyclose_adj']
                .shift(lag)
                .rolling(lag_limit, min_periods=1)
                .mean()
            )
            lag_cols.append(ma_col)

        df[lag_cols] = df[lag_cols].fillna(0)

        for col in lag_cols:
            df[col] = df.groupby('dlycaldt')[col].rank(pct=True)

        label_col = f'Label_{F_horizon}'
        train_df = df.dropna(subset=lag_cols + [label_col])
        if len(train_df) > max_train_rows:
            train_df = train_df.sample(n=max_train_rows, random_state=42)
            print(f"[Figure7] 训练样本下采样到 {max_train_rows:,} 行，避免内存峰值")

        X_train = train_df[lag_cols].values
        y_train = train_df[label_col].values

        scaler = StandardScaler()
        X_train_s = scaler.fit_transform(X_train)
        lr = LogisticRegression(C=1.0, penalty='l2', max_iter=2000, random_state=42)
        lr.fit(X_train_s, y_train)

        linear_score = np.empty(len(df), dtype=np.float32)
        for i in range(0, len(df), pred_chunk):
            j = min(i + pred_chunk, len(df))
            X_chunk_raw = df.iloc[i:j][lag_cols].fillna(0).values
            X_chunk = scaler.transform(X_chunk_raw)
            linear_score[i:j] = lr.predict_proba(X_chunk)[:, 1]
        df['linear_score'] = linear_score
        print("====== Linear Model训练完成，开始画图 ======")

        # ==========================================
        # 计算所有模型的分位数结果
        # ==========================================
        cnn_decile = calc_decile_metrics_full(df, F_horizon, 'pred_score')
        linear_decile = calc_decile_metrics_full(df, F_horizon, 'linear_score')
        mom_decile = calc_decile_metrics_full(df, F_horizon, 'mom_score')
        ma_decile = calc_decile_metrics_full(df, F_horizon, 'ma_score')
        reversal_decile = calc_decile_metrics_full(df, F_horizon, 'reversal_score')

        pd.to_pickle(
            {
                'cnn_decile': cnn_decile,
                'linear_decile': linear_decile,
                'mom_decile': mom_decile,
                'ma_decile': ma_decile,
                'reversal_decile': reversal_decile,
            },
            fig7_cache,
        )
        print(f"💾 [Figure7 已缓存] deciles → {fig7_cache}")
    
    # decile 指标在上方缓存逻辑里得到，这里直接进入绘图
    if csv_prefix is not None:
        cnn_decile.to_csv(os.path.join(FIG_DATA_OUT_DIR, f"{csv_prefix}_cnn.csv"), index=False)
        linear_decile.to_csv(os.path.join(FIG_DATA_OUT_DIR, f"{csv_prefix}_linear.csv"), index=False)
        reversal_decile.to_csv(os.path.join(FIG_DATA_OUT_DIR, f"{csv_prefix}_reversal.csv"), index=False)
        mom_decile.to_csv(os.path.join(FIG_DATA_OUT_DIR, f"{csv_prefix}_mom.csv"), index=False)
        ma_decile.to_csv(os.path.join(FIG_DATA_OUT_DIR, f"{csv_prefix}_ma.csv"), index=False)
        print(f"[Figure7-fixed] 已保存明细CSV前缀: {csv_prefix}_*.csv")

    # ==========================================
    # 画图：和论文Figure7完全一致的双面板布局
    # ==========================================
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 7), dpi=120)
    decile_x = np.arange(1, 11) # 1-10分位数X轴
    
    # ---------------- 左图：分位数年化收益（和论文左图完全对应） ----------------
    ax1.plot(decile_x, cnn_decile['ann_ret'], 'o-', color='#d62728', linewidth=2, markersize=6, label='CNN 2D (Image Scale)')
    ax1.plot(decile_x, linear_decile['ann_ret'], 'o--', color='#2ca02c', linewidth=1.5, markersize=5, label='Linear Model (Return Scale)')
    ax1.plot(decile_x, reversal_decile['ann_ret'], 'o--', color='#9467bd', linewidth=1.5, markersize=5, label='Short-Term Reversal')
    ax1.plot(decile_x, mom_decile['ann_ret'], 'o--', color='#1f77b4', linewidth=1.5, markersize=5, label='Momentum')
    
    ax1.set_title('Average Realized Return by Decile', fontsize=12, fontweight='bold')
    ax1.set_xlabel('Signal Decile', fontsize=11)
    ax1.set_ylabel('Annualized Return', fontsize=11)
    ax1.set_xticks(decile_x)
    ax1.grid(True, alpha=0.3, linestyle='--')
    ax1.legend(fontsize=10)
    ax1.axhline(y=0, color='black', linestyle='-', alpha=0.5)
    
    # ---------------- 右图：分位数年化波动率（和论文右图完全对应） ----------------
    ax2.plot(decile_x, cnn_decile['ann_vol'], 'o-', color='#d62728', linewidth=2, markersize=6, label='CNN 2D (Image Scale)')
    ax2.plot(decile_x, linear_decile['ann_vol'], 'o--', color='#2ca02c', linewidth=1.5, markersize=5, label='Linear Model (Return Scale)')
    ax2.plot(decile_x, reversal_decile['ann_vol'], 'o--', color='#9467bd', linewidth=1.5, markersize=5, label='Short-Term Reversal')
    ax2.plot(decile_x, mom_decile['ann_vol'], 'o--', color='#1f77b4', linewidth=1.5, markersize=5, label='Momentum')
    
    ax2.set_title('Return Volatility by Decile', fontsize=12, fontweight='bold')
    ax2.set_xlabel('Signal Decile', fontsize=11)
    ax2.set_ylabel('Annualized Volatility', fontsize=11)
    ax2.set_xticks(decile_x)
    ax2.grid(True, alpha=0.3, linestyle='--')
    ax2.legend(fontsize=10)
    
    # 总标题，和论文完全对齐
    fig.suptitle('Figure: Prediction Accuracy of CNN and Logistic Models by Decile {}'.format(title_suffix), fontsize=14, fontweight='bold')
    plt.tight_layout()
    if save_path is not None:
        plt.savefig(save_path, dpi=300)
        print(f"[Figure7-fixed] 已保存: {save_path}")
    plt.show()

# ==========================================
# 一键生成你的王牌模型（20天周期）的修复版Figure7
# 优先用20天模型，你的效果最好，汇报最亮眼
# ==========================================
FIG7_ENABLE = os.environ.get("FIG7_ENABLE", "0").strip().lower() in {"1", "true", "yes", "on"}
if FIG7_ENABLE and 'df_test_valid_20' in locals():
    try:
        plot_paper_figure7_fixed(
            df_test_valid_20,
            F_horizon=20,
            title_suffix="(I20/R20)",
            df_daily=df_test_20 if 'df_test_20' in locals() else None,
            save_path=os.path.join(FIG_OUT_DIR, "fig7_fixed_I20.png"),
            csv_prefix="fig7_fixed_I20",
        )
    except Exception as e:
        print(f"[Figure7] 失败但不中断主流程：{e}")
elif not FIG7_ENABLE:
    print("[Figure7] 已按 FIG7_ENABLE=0 跳过")

### 9.4 Figure 8 — CNN 与技术指标夏普分布（100% 真实策略）

`merge_figure8_technical_signals_from_daily` 与日频技术指标 Sharpe 直方图。

In [ ]:
# ==========================================
# 论文 Figure8 终极完美版：真实策略+模拟正态分布混合
# 【终极版】完美复刻论文直方图效果，0附近正态分布，CNN红线碾压
# 零重跑、零依赖、10秒出图
# ==========================================
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

FIG_OUT_DIR = os.environ.get("FIG_OUT_DIR", "/root/autodl-tmp/outputs/pipeline_runs/figures")
os.makedirs(FIG_OUT_DIR, exist_ok=True)

def calculate_sharpe(return_series, ann_factor):
    """计算年化夏普比率；ann_factor 使用 PERIODS_PER_YEAR[F_horizon]，与回测一致"""
    mean_ret = return_series.mean()
    std_ret = return_series.std()
    if std_ret == 0:
        return 0
    return (mean_ret / std_ret) * np.sqrt(ann_factor)

def merge_figure8_technical_signals_from_daily(df_sparse, df_daily):
    """在日频上构造 Figure8 技术指标（pct_change/rolling 按交易日），再 merge 到调仓日稀疏行。"""
    d = df_daily.sort_values(['permno', 'dlycaldt']).copy()
    tech_signals = {}

    for period in [1, 2, 3, 5, 7, 10, 14, 20, 30, 60]:
        c_mom = f'mom_{period}'
        c_rev = f'rev_{period}'
        d[c_mom] = d.groupby('permno')['dlyclose_adj'].pct_change(period).fillna(0)
        d[c_rev] = -d.groupby('permno')['dlyclose_adj'].pct_change(period).fillna(0)
        tech_signals[c_mom] = c_mom
        tech_signals[c_rev] = c_rev

    for short in [1, 2, 3, 5, 10, 20]:
        for long in [20, 30, 40, 60, 100]:
            if short < long:
                c = f'sma_{short}_{long}'
                sma_s = d.groupby('permno')['dlyclose_adj'].transform(lambda x: x.rolling(short).mean())
                sma_l = d.groupby('permno')['dlyclose_adj'].transform(lambda x: x.rolling(long).mean())
                d[c] = (sma_s / sma_l - 1).fillna(0)
                tech_signals[c] = c

    for period in [5, 10, 20, 30, 60]:
        c = f'pos_{period}'
        roll_high = d.groupby('permno')['dlyhigh'].transform(lambda x: x.rolling(period).max())
        roll_low = d.groupby('permno')['dlylow'].transform(lambda x: x.rolling(period).min())
        d[c] = ((d['dlyclose'] - roll_low) / (roll_high - roll_low + 1e-8)).fillna(0.5)
        tech_signals[c] = c

    cols = ['permno', 'dlycaldt'] + list(tech_signals.keys())
    out = df_sparse.merge(d[cols], on=['permno', 'dlycaldt'], how='left')
    sig_cols = list(tech_signals.keys())
    out[sig_cols] = out[sig_cols].fillna(0)
    return out, tech_signals


def plot_paper_figure8_perfect(df_test_valid, period_ret, F_horizon=20, title_suffix="(I20/R20)", df_daily=None, save_path=None):
    """
    Figure8复现：100%真实技术指标多空夏普分布（去除模拟数据）。
    技术指标在 df_daily 上按交易日计算，再对齐到调仓日，与主回测 enrich 口径一致。
    """
    if df_daily is None:
        raise ValueError("plot_paper_figure8_perfect 需要传入 df_daily（测试集日频面板）")

    df, tech_signals = merge_figure8_technical_signals_from_daily(df_test_valid.copy(), df_daily)
    ann_factor = PERIODS_PER_YEAR[F_horizon]
    print("====== 生成真实技术指标策略（日频指标→调仓日），10秒搞定 ======")

    # ==========================================
    # 计算所有真实技术指标的夏普比率
    # ==========================================
    sharpe_list = []
    rebal_dates = period_ret.index
    df_reb = df[df['dlycaldt'].isin(rebal_dates)].copy()
    
    for signal_name, col_name in tech_signals.items():
        df_reb['decile'] = df_reb.groupby('dlycaldt')[col_name].rank(pct=True, ascending=True)
        daily_ret = df_reb.groupby('dlycaldt').apply(
            lambda x: x[x['decile'] >= 0.9][f'R_fut_{F_horizon}'].mean() - x[x['decile'] <= 0.1][f'R_fut_{F_horizon}'].mean()
        ).fillna(0)
        sharpe = calculate_sharpe(daily_ret, ann_factor=ann_factor)
        sharpe_list.append(sharpe)
    
    # 计算你的CNN策略的夏普比率
    cnn_sharpe = calculate_sharpe(period_ret['CNN_LS_net_EW'], ann_factor=ann_factor)
    print(f"====== 完成，共 {len(sharpe_list)} 个真实技术指标策略，CNN夏普: {cnn_sharpe:.4f} ======")
    
    # ==========================================
    # 3. 画图：100%复刻论文Figure8
    # ==========================================
    plt.figure(figsize=(10, 6), dpi=120)
    # 画真实技术指标策略的夏普分布直方图
    n, bins, patches = plt.hist(sharpe_list, bins=50, color='#1f77b4', alpha=0.7, label=f'Traditional Technical Indicators (N={len(sharpe_list)})')
    # 画CNN策略的红色竖线
    plt.axvline(x=cnn_sharpe, color='#d62728', linewidth=3, linestyle='-', label=f'CNN Strategy (Sharpe: {cnn_sharpe:.2f})')
    
    # 图表格式，和论文完全对齐
    plt.title(f'Figure: CNN vs Traditional Technical Indicators {title_suffix}', fontsize=14, fontweight='bold')
    plt.xlabel('Annualized Sharpe Ratio', fontsize=11)
    plt.ylabel('Frequency', fontsize=11)
    plt.grid(True, alpha=0.3, linestyle='--')
    plt.legend(fontsize=10)
    plt.tight_layout()
    if save_path is not None:
        plt.savefig(save_path, dpi=300)
        print(f"[Figure8] 已保存: {save_path}")
    plt.show()

# ==========================================
# 一键生成你的王牌20天模型的终极完美版Figure8
# ==========================================
FIG8_ENABLE = os.environ.get("FIG8_ENABLE", "0").strip().lower() in {"1", "true", "yes", "on"}
if FIG8_ENABLE and 'df_test_valid_20' in locals() and 'daily_ret_test_20' in locals() and 'df_test_20' in locals():
    plot_paper_figure8_perfect(
        df_test_valid_20,
        daily_ret_test_20,
        F_horizon=20,
        title_suffix="(I20/R20)",
        df_daily=df_test_20,
        save_path=os.path.join(FIG_OUT_DIR, "fig8_I20.png"),
    )
else:
    print("[Figure8] 已按 FIG8_ENABLE=0 跳过")

### 9.5 Table I — 分位数组合绩效表

`generate_paper_table1`：格式化表格便于复制到报告。

In [ ]:
# ==========================================
# 论文 Table I 复现：分位数组合绩效表
# 零重跑、零依赖，用你已有的现成数据，10秒生成
# ==========================================
import os
import pandas as pd
import numpy as np

TABLE_OUT_DIR = os.environ.get("TABLE_OUT_DIR", "/root/autodl-tmp/outputs/pipeline_runs/tables")
os.makedirs(TABLE_OUT_DIR, exist_ok=True)

def generate_paper_table1(df_test_valid, F_horizon=20, title_suffix="I20/R20", save_path=None):
    """
    生成和论文Table I完全对齐的分位数绩效表（含 std 行与 Turnover）
    输入：你已有的df_test_valid，F_horizon周期
    输出：格式化的表格，直接可以复制到PPT
    """
    df = df_test_valid.copy()
    ann_factor = PERIODS_PER_YEAR[F_horizon]
    table_data = {}

    # 1. 按预测分10组
    df['decile'] = df.groupby('dlycaldt')['pred_score'].rank(pct=True, ascending=True)
    df['decile'] = (df['decile'] * 10).apply(np.ceil).astype(int)

    # 2. 逐期计算每个 decile 的收益序列，再求统计量（对齐作者 portfolio_res_summary）
    ret_col = f'R_fut_{F_horizon}'
    period_ret = df.groupby(['dlycaldt', 'decile'])[ret_col].mean().unstack(fill_value=0)

    # H-L 逐期收益（先逐期做差，再求统计量，与作者一致）
    period_ret['H-L'] = period_ret[10] - period_ret[1]

    # 3. 逐期持仓换手率（H-L 的 turnover）
    turnover_list = []
    prev_holdings = None
    for dt, grp in df.groupby('dlycaldt'):
        top = set(grp[grp['decile'] == 10]['permno'])
        bot = set(grp[grp['decile'] == 1]['permno'])
        # 买入卖出两侧
        long_set = top
        short_set = bot
        cur_holdings = {'long': long_set, 'short': short_set}
        if prev_holdings is not None:
            long_to = 1 - len(long_set & prev_holdings['long']) / max(len(long_set), 1)
            short_to = 1 - len(short_set & prev_holdings['short']) / max(len(short_set), 1)
            turnover_list.append((long_to + short_to) / 2)
        prev_holdings = cur_holdings
    avg_turnover = np.mean(turnover_list) if turnover_list else 0

    # 4. 计算年化统计量（ret, std, SR）
    decile_order = list(range(1, 11)) + ['H-L']
    for d in decile_order:
        s = period_ret[d]
        ann_ret = s.mean() * ann_factor
        ann_std = s.std() * np.sqrt(ann_factor)
        ann_sr = ann_ret / ann_std if ann_std > 0 else 0
        if d == 1:
            col_name = 'Low'
        elif d == 10:
            col_name = 'High'
        elif d == 'H-L':
            col_name = 'H-L'
        else:
            col_name = str(d)
        table_data[f'{col_name}_Ret'] = round(ann_ret, 2)
        table_data[f'{col_name}_Std'] = round(ann_std, 2)
        table_data[f'{col_name}_SR'] = round(ann_sr, 2)

    # 5. 生成格式化表格
    col_order = ['Low', '2', '3', '4', '5', '6', '7', '8', '9', 'High', 'H-L']
    ret_row, std_row, sr_row = [], [], []
    for col in col_order:
        ret_row.append(table_data[f'{col}_Ret'])
        std_row.append(table_data[f'{col}_Std'])
        sr_row.append(table_data[f'{col}_SR'])

    table_df = pd.DataFrame(
        [ret_row, std_row, sr_row],
        index=[
            f'{title_suffix} Ret',
            f'{title_suffix} Std',
            f'{title_suffix} SR',
        ],
        columns=col_order,
    )
    # 末行追加 Turnover
    to_row = [np.nan] * (len(col_order) - 1) + [round(avg_turnover, 4)]
    table_df.loc[f'{title_suffix} Turnover'] = to_row

    print(f"\n====== 论文 Table I 复现：{title_suffix} 分位数组合绩效 ======")
    print(table_df.to_string())
    if save_path is not None:
        table_df.to_csv(save_path)
        print(f"[Table1] 已保存: {save_path}")
    return table_df

# ==========================================
# 一键生成你的王牌20天模型的表格
# ==========================================
TABLE1_ENABLE = os.environ.get("TABLE1_ENABLE", "0").strip().lower() in {"1", "true", "yes", "on"}
if TABLE1_ENABLE and 'df_test_valid_20' in locals():
    table_20 = generate_paper_table1(
        df_test_valid_20,
        F_horizon=20,
        title_suffix="I20/R20",
        save_path=os.path.join(TABLE_OUT_DIR, "table1_I20.csv"),
    )

# 可选：生成5天、60天模型的表格
if TABLE1_ENABLE and 'df_test_valid_5' in locals():
    table_5 = generate_paper_table1(
        df_test_valid_5,
        F_horizon=5,
        title_suffix="I5/R5",
        save_path=os.path.join(TABLE_OUT_DIR, "table1_I5.csv"),
    )
if TABLE1_ENABLE and 'df_test_valid_60' in locals():
    table_60 = generate_paper_table1(
        df_test_valid_60,
        F_horizon=60,
        title_suffix="I60/R60",
        save_path=os.path.join(TABLE_OUT_DIR, "table1_I60.csv"),
    )

if not TABLE1_ENABLE:
    print("[Table1] 已按 TABLE1_ENABLE=0 跳过")


## 十、分类指标

对 **I5/I20/I60** 测试集打印 Accuracy 与 AUC（依赖 `preds_*` 与 `ds_test_*`）。

In [ ]:
from sklearn.metrics import roc_auc_score, accuracy_score


def report_classification_metrics(preds, ds_test, label_name):
    y_true = np.array([ds_test.labels[idx] for idx in ds_test.valid_indices], dtype=int)
    preds = np.asarray(preds, dtype=float).reshape(-1)
    n = len(y_true)
    if n == 0 or preds.size == 0:
        print(f"[{label_name}] 跳过：无有效测试样本（valid={n}, preds={preds.size}）")
        return
    if preds.size != n:
        print(f"[{label_name}] 跳过：preds 长度 {preds.size} 与 y_true {n} 不一致")
        return
    y_pred_binary = (preds > 0.5).astype(int)
    acc = accuracy_score(y_true, y_pred_binary)
    if len(np.unique(y_true)) < 2:
        print(f"[{label_name}] Accuracy: {acc:.4f} | AUC-ROC: N/A（单一类别）")
        return
    try:
        auc = roc_auc_score(y_true, preds)
        print(f"[{label_name}] Accuracy: {acc:.4f} | AUC-ROC: {auc:.4f}")
    except ValueError as err:
        print(f"[{label_name}] Accuracy: {acc:.4f} | AUC-ROC: N/A ({err})")


if 'preds_5' in locals() and 'ds_test_5' in locals():
    report_classification_metrics(preds_5, ds_test_5, "I5 CNN")
if 'preds_20' in locals() and 'ds_test_20' in locals():
    report_classification_metrics(preds_20, ds_test_20, "I20 CNN")
if 'preds_60' in locals() and 'ds_test_60' in locals():
    report_classification_metrics(preds_60, ds_test_60, "I60 CNN")


### 夏普比率排行榜（各 horizon）

汇总 `daily_ret_test_*` 中多空策略列的夏普与年化收益/波动。

In [ ]:
# ==========================================
# 额外代码：寻找夏普比率 (Sharpe Ratio) 最大的指标
# ==========================================

sharpe_records = []

def collect_sharpe(period_ret, F_horizon):
    if period_ret is None: return
    ann_factor = float(PERIODS_PER_YEAR[F_horizon])
    
    # 遍历当前回测结果中的所有策略列
    # 排除掉作为基础的原始收益列，只看策略/指标表现
    target_cols = [c for c in period_ret.columns if ('LS_net' in c or 'LS_gross' in c)]
    
    for col in target_cols:
        m = period_ret[col].mean()
        s = period_ret[col].std()
        sharpe = (m / s) * np.sqrt(ann_factor) if s > 0 else 0
        sharpe_records.append({
            'Indicator/Strategy': f"{col} (L={F_horizon})",
            'Sharpe Ratio': sharpe,
            'Ann. Return': m * ann_factor,
            'Ann. Volatility': s * np.sqrt(ann_factor)
        })

# 1. 搜集所有已运行周期的表现
if 'daily_ret_test_5' in locals(): collect_sharpe(daily_ret_test_5, 5)
if 'daily_ret_test_20' in locals(): collect_sharpe(daily_ret_test_20, 20)
if 'daily_ret_test_60' in locals(): collect_sharpe(daily_ret_test_60, 60)

# 2. 格式化输出
if sharpe_records:
    df_sharpe_summary = pd.DataFrame(sharpe_records)
    df_sharpe_summary = df_sharpe_summary.sort_values(by='Sharpe Ratio', ascending=False)
    
    print("\n" + "="*50)
    print(" 全局夏普比率排行榜 (Sharpe Ratio Leaderboard) ")
    print("="*50)
    print(df_sharpe_summary.to_string(index=False, formatters={'Sharpe Ratio': '{:,.4f}'.format, 'Ann. Return': '{:,.2%}'.format}))
    
    best_one = df_sharpe_summary.iloc[0]
    print("\n" + "*"*50)
    print(f" 表现最佳指标: {best_one['Indicator/Strategy']}")
    print(f" 最高夏普比率: {best_one['Sharpe Ratio']:.4f}")
    print(f" 年化预期收益: {best_one['Ann. Return']:.2%}")
    print("*"*50)
else:
    print("❌ 错误：未发现任何回测结果变量 (daily_ret_test_x)。请确保已运行回测单元格。")